In [1]:
# CELL 1 — Mount Drive and basic paths

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import json
import ast
import time
import gc
import random
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)

DATA_DIR = Path("/content/drive/MyDrive/Data")
MOVIES_PATH = DATA_DIR / "movies_metadata.csv"
CREDITS_PATH = DATA_DIR / "credits.csv"

print("Python:", sys.version)
print("Movies exists:", MOVIES_PATH.exists(), MOVIES_PATH)
print("Credits exists:", CREDITS_PATH.exists(), CREDITS_PATH)

Mounted at /content/drive
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Movies exists: True /content/drive/MyDrive/Data/movies_metadata.csv
Credits exists: True /content/drive/MyDrive/Data/credits.csv


In [ ]:
# CELL 2 — Install required libraries

!pip -q install -U sentence-transformers faiss-cpu transformers accelerate bitsandbytes datasets ragas

In [ ]:
# CELL 3 — Load CSV files and inspect

import pandas as pd

movies_raw = pd.read_csv(MOVIES_PATH, low_memory=False)
credits_raw = pd.read_csv(CREDITS_PATH)

print("movies_metadata.csv shape:", movies_raw.shape)
print("credits.csv shape:", credits_raw.shape)

print("\nMovies columns:")
print(movies_raw.columns.tolist())

print("\nCredits columns:")
print(credits_raw.columns.tolist())

print("\nMovies sample:")
display(movies_raw.head(3))

print("\nCredits sample:")
display(credits_raw.head(3))

movies_metadata.csv shape: (45466, 24)
credits.csv shape: (45476, 3)

Movies columns:
['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count']

Credits columns:
['cast', 'crew', 'id']

Movies sample:


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0



Credits sample:


,cast,crew,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...",15602


In [ ]:
# CELL 4 — Preprocess movies and credits

import numpy as np

movies = movies_raw.copy()
credits = credits_raw.copy()

print("Initial movies:", len(movies))
print("Initial credits:", len(credits))

# 1) Clean movie id column
# movies_metadata.csv içinde bazı bozuk id satırları var; sadece numeric id tutuyoruz.
movies = movies[movies["id"].astype(str).str.match(r"^\d+$", na=False)].copy()
movies["id"] = movies["id"].astype(int)
credits["id"] = credits["id"].astype(int)

print("After valid numeric id:", len(movies))

# 2) Drop rows with missing title
before = len(movies)
movies = movies.dropna(subset=["title"]).copy()
print("Dropped missing title:", before - len(movies))

# 3) Drop rows with missing or empty overview
before = len(movies)
movies["overview"] = movies["overview"].fillna("").astype(str).str.strip()
movies = movies[movies["overview"].str.len() > 0].copy()
print("Dropped empty overview:", before - len(movies))

# 4) Parse release year
movies["release_year"] = pd.to_datetime(
    movies["release_date"],
    errors="coerce"
).dt.year

# 5) Parse genres from stringified JSON-like list
def parse_genres(x):
    try:
        items = ast.literal_eval(x) if isinstance(x, str) else []
        return [item.get("name", "") for item in items if isinstance(item, dict) and item.get("name")]
    except Exception:
        return []

movies["genres_list"] = movies["genres"].apply(parse_genres)
movies["genres_text"] = movies["genres_list"].apply(lambda xs: ", ".join(xs) if xs else "Unknown")

# 6) Parse director and top cast from credits
def parse_director(crew_str):
    try:
        crew = ast.literal_eval(crew_str) if isinstance(crew_str, str) else []
        directors = [
            person.get("name", "")
            for person in crew
            if isinstance(person, dict) and person.get("job") == "Director" and person.get("name")
        ]
        return directors[0] if directors else "Unknown"
    except Exception:
        return "Unknown"

def parse_top_cast(cast_str, max_cast=5):
    try:
        cast = ast.literal_eval(cast_str) if isinstance(cast_str, str) else []
        names = [
            person.get("name", "")
            for person in cast[:max_cast]
            if isinstance(person, dict) and person.get("name")
        ]
        return ", ".join(names) if names else "Unknown"
    except Exception:
        return "Unknown"

credits["director"] = credits["crew"].apply(parse_director)
credits["cast_text"] = credits["cast"].apply(parse_top_cast)

# 7) Merge movies + credits
movies = movies.merge(
    credits[["id", "director", "cast_text"]],
    on="id",
    how="left"
)

movies["director"] = movies["director"].fillna("Unknown")
movies["cast_text"] = movies["cast_text"].fillna("Unknown")

print("After merge:", len(movies))

# 8) Drop duplicate titles, keep the one with longer overview
movies["_overview_len_chars"] = movies["overview"].str.len()
movies = movies.sort_values("_overview_len_chars", ascending=False)

before = len(movies)
movies = movies.drop_duplicates(subset=["title"], keep="first").copy()
print("Dropped duplicate titles:", before - len(movies))

# 9) Create final document_text
def make_document_text(row):
    year = "Unknown" if pd.isna(row["release_year"]) else str(int(row["release_year"]))
    return (
        f"Title: {row['title']}\n"
        f"Year: {year}\n"
        f"Genres: {row['genres_text']}\n"
        f"Director: {row['director']}\n"
        f"Cast: {row['cast_text']}\n"
        f"Overview: {row['overview']}"
    )

movies["document_text"] = movies.apply(make_document_text, axis=1)

# 10) Reset index so FAISS ids match DataFrame row positions
movies = movies.reset_index(drop=True)

print("\nFinal preprocessed corpus size:", len(movies))

display(movies[[
    "id", "title", "release_year", "genres_text", "director", "cast_text", "overview", "document_text"
]].head(3))

Initial movies: 45466
Initial credits: 45476
After valid numeric id: 45463
Dropped missing title: 3
Dropped empty overview: 959
After merge: 44577
Dropped duplicate titles: 3210

Final preprocessed corpus size: 41367


,id,title,release_year,genres_text,director,cast_text,overview,document_text
0,174271,The Fortunes and Misfortunes of Moll Flanders,1996.0,"Comedy, Drama, Romance",David Attwood,"James Bowers, Alex Kingston, Geoffrey Beevers,...",In her filthy cell in Newgate prison Moll Flan...,Title: The Fortunes and Misfortunes of Moll Fl...
1,279966,The Forgotten,2014.0,"Horror, Thriller",Oliver Frampton,"Shaun Dingwall, Clem Tibber, Elarica Gallacher...",When a father and son are forced to squat in a...,Title: The Forgotten\nYear: 2014\nGenres: Horr...
2,25868,Aladin,2009.0,"Fantasy, Drama, Romance, Foreign",Sujoy Ghosh,"Amitabh Bachchan, Sanjay Dutt, Ritesh Deshmukh...","Based in the municipality of Khwaish, abused b...","Title: Aladin\nYear: 2009\nGenres: Fantasy, Dr..."


In [ ]:
# CELL 5 — Corpus statistics for Analysis Report

# Simple token estimate: whitespace split
movies["overview_token_count"] = movies["overview"].apply(lambda x: len(str(x).split()))
movies["document_token_count"] = movies["document_text"].apply(lambda x: len(str(x).split()))

overview_stats = {
    "mean": movies["overview_token_count"].mean(),
    "median": movies["overview_token_count"].median(),
    "max": movies["overview_token_count"].max()
}

document_stats = {
    "mean": movies["document_token_count"].mean(),
    "median": movies["document_token_count"].median(),
    "max": movies["document_token_count"].max()
}

print("Final corpus size:", len(movies))

print("\nOverview token statistics:")
print(overview_stats)

print("\nDocument_text token statistics:")
print(document_stats)

print("\nExample document_text strings:")
for i in [0, 10, 100]:
    print("=" * 80)
    print(movies.loc[i, "document_text"][:1500])

Final corpus size: 41367

Overview token statistics:
{'mean': np.float64(56.379771315299635), 'median': 50.0, 'max': 187}

Document_text token statistics:
{'mean': np.float64(79.87714845166437), 'median': 73.0, 'max': 212}

Example document_text strings:
Title: The Fortunes and Misfortunes of Moll Flanders
Year: 1996
Genres: Comedy, Drama, Romance
Director: David Attwood
Cast: James Bowers, Alex Kingston, Geoffrey Beevers, Lucy Evans, Anthony Bessick
Overview: In her filthy cell in Newgate prison Moll Flanders, dubbed 'the wickedest woman in England' tells her story. Born in the gaol, after her mother is transported Moll is raised by the kindly mayor of Colchester and his wife, whose two sons lust after her. She enjoys sex with handsome Rowland, who teaches her that money talks, but, realizing he only wants her as a mistress, she marries his duller brother Robin, who conveniently dies after five years, leaving her wealthy. She goes to London, briefly meeting highwayman Jemmy Seagrove, 

In [ ]:
# CELL 6 — Load embedding model

import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"

print("Using device:", device)
print("Loading embedding model...")

embed_model = SentenceTransformer(
    EMBED_MODEL_NAME,
    device=device
)

print("Embedding model loaded:", EMBED_MODEL_NAME)

Using device: cuda
Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded: BAAI/bge-small-en-v1.5


In [ ]:
# CELL 7 — Encode all documents with bi-encoder

texts = movies["document_text"].tolist()

BATCH_SIZE = 128

start_time = time.perf_counter()

doc_embeddings = embed_model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

encoding_time = time.perf_counter() - start_time
docs_per_sec = len(texts) / encoding_time

doc_embeddings = doc_embeddings.astype("float32")

print("Embedding shape:", doc_embeddings.shape)
print(f"Encoding time: {encoding_time:.2f} seconds")
print(f"Throughput: {docs_per_sec:.2f} docs/sec")
print("Embedding dtype:", doc_embeddings.dtype)
print("First vector norm:", np.linalg.norm(doc_embeddings[0]))

Batches:   0%|          | 0/324 [00:00<?, ?it/s]

Embedding shape: (41367, 384)
Encoding time: 153.71 seconds
Throughput: 269.13 docs/sec
Embedding dtype: float32
First vector norm: 1.0


In [ ]:
# CELL 8 — Build FAISS index and run sanity check

import faiss

embedding_dim = doc_embeddings.shape[1]

# Since embeddings are normalized, IndexFlatIP gives cosine similarity.
faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(doc_embeddings)

print("FAISS index created.")
print("Embedding dimension:", embedding_dim)
print("Number of vectors in index:", faiss_index.ntotal)

def faiss_search(query, k=5):
    query_embedding = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = faiss_index.search(query_embedding, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        row = movies.iloc[int(idx)]
        results.append({
            "faiss_id": int(idx),
            "title": row["title"],
            "year": row["release_year"],
            "score": float(score),
            "director": row["director"],
            "document_text": row["document_text"]
        })

    return results

# Sanity check
query = "Who directed Toy Story?"
results = faiss_search(query, k=5)

for i, r in enumerate(results, start=1):
    print(f"{i}. {r['title']} ({r['year']}) | score={r['score']:.4f} | director={r['director']}")

FAISS index created.
Embedding dimension: 384
Number of vectors in index: 41367
1. Toy Story 3 (2010.0) | score=0.7242 | director=Lee Unkrich
2. Toy Story 2 (1999.0) | score=0.7075 | director=John Lasseter
3. Toy Story (1995.0) | score=0.6955 | director=John Lasseter
4. Toy Story of Terror! (2013.0) | score=0.6739 | director=Angus MacLane
5. The Pixar Story (2007.0) | score=0.6688 | director=Leslie Iwerks


In [ ]:
# CELL 8B — Fix faiss_search by adding cast_text and genres_text

def faiss_search(query, k=5):
    query_embedding = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = faiss_index.search(query_embedding, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        row = movies.iloc[int(idx)]
        results.append({
            "faiss_id": int(idx),
            "title": row["title"],
            "year": row["release_year"],
            "score": float(score),
            "director": row["director"],
            "cast_text": row["cast_text"],
            "genres_text": row["genres_text"],
            "overview": row["overview"],
            "document_text": row["document_text"]
        })

    return results

In [ ]:
# CELL 8C — Save embeddings and FAISS index to Drive

INDEX_DIR = Path("/content/drive/MyDrive/movie_rag_faiss")
INDEX_DIR.mkdir(parents=True, exist_ok=True)

np.save(INDEX_DIR / "doc_embeddings_bge_small.npy", doc_embeddings)
faiss.write_index(faiss_index, str(INDEX_DIR / "faiss_bge_small.index"))

movies.to_pickle(INDEX_DIR / "movies_preprocessed.pkl")

print("Saved embeddings, FAISS index, and preprocessed movies to:", INDEX_DIR)

Saved embeddings, FAISS index, and preprocessed movies to: /content/drive/MyDrive/movie_rag_faiss


In [ ]:
# CELL 9 — Load cross-encoder re-ranker

from sentence_transformers import CrossEncoder

RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

print("Loading re-ranker...")
reranker = CrossEncoder(
    RERANKER_MODEL_NAME,
    device=device
)

print("Re-ranker loaded:", RERANKER_MODEL_NAME)

Loading re-ranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Re-ranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


In [ ]:
# CELL 10 — Retrieval and re-ranking functions

def retrieve(query, k=20):
    return faiss_search(query, k=k)

def rerank(query, candidates, n=5):
    if not candidates:
        return []

    pairs = [(query, c["document_text"]) for c in candidates]

    scores = reranker.predict(pairs)

    reranked = []
    for candidate, score in zip(candidates, scores):
        item = candidate.copy()
        item["rerank_score"] = float(score)
        reranked.append(item)

    reranked = sorted(
        reranked,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return reranked[:n]

# Sanity check with the sequel-confusion query
query = "Who directed Toy Story?"

candidates = retrieve(query, k=20)
reranked_results = rerank(query, candidates, n=5)

print("Bi-encoder top-5:")
for i, r in enumerate(candidates[:5], start=1):
    print(f"{i}. {r['title']} ({r['year']}) | bi_score={r['score']:.4f} | director={r['director']}")

print("\nAfter re-ranking top-5:")
for i, r in enumerate(reranked_results, start=1):
    print(
        f"{i}. {r['title']} ({r['year']}) | "
        f"bi_score={r['score']:.4f} | rerank_score={r['rerank_score']:.4f} | "
        f"director={r['director']}"
    )

Bi-encoder top-5:
1. Toy Story 3 (2010.0) | bi_score=0.7242 | director=Lee Unkrich
2. Toy Story 2 (1999.0) | bi_score=0.7075 | director=John Lasseter
3. Toy Story (1995.0) | bi_score=0.6955 | director=John Lasseter
4. Toy Story of Terror! (2013.0) | bi_score=0.6739 | director=Angus MacLane
5. The Pixar Story (2007.0) | bi_score=0.6688 | director=Leslie Iwerks

After re-ranking top-5:
1. Toy Story (1995.0) | bi_score=0.6955 | rerank_score=7.9102 | director=John Lasseter
2. Toy Story 2 (1999.0) | bi_score=0.7075 | rerank_score=7.8594 | director=John Lasseter
3. Toy Story 3 (2010.0) | bi_score=0.7242 | rerank_score=7.4759 | director=Lee Unkrich
4. Toy Story That Time Forgot (2014.0) | bi_score=0.6582 | rerank_score=6.9290 | director=Steve Purcell
5. Toy Story of Terror! (2013.0) | bi_score=0.6739 | rerank_score=6.7335 | director=Angus MacLane


In [ ]:
# CELL 11 — Show bi-encoder vs re-ranker for 3 example queries

comparison_queries = [
    "Who directed Toy Story?",
    "What is the plot of The Godfather?",
    "Who directed Alien?"
]

for query in comparison_queries:
    print("=" * 100)
    print("QUERY:", query)

    candidates = retrieve(query, k=20)
    reranked_results = rerank(query, candidates, n=5)

    print("\nBi-encoder top-5:")
    for i, r in enumerate(candidates[:5], start=1):
        print(f"{i}. {r['title']} ({r['year']}) | bi_score={r['score']:.4f} | director={r['director']}")

    print("\nAfter re-ranking top-5:")
    for i, r in enumerate(reranked_results, start=1):
        print(
            f"{i}. {r['title']} ({r['year']}) | "
            f"bi_score={r['score']:.4f} | rerank_score={r['rerank_score']:.4f} | "
            f"director={r['director']}"
        )

QUERY: Who directed Toy Story?

Bi-encoder top-5:
1. Toy Story 3 (2010.0) | bi_score=0.7242 | director=Lee Unkrich
2. Toy Story 2 (1999.0) | bi_score=0.7075 | director=John Lasseter
3. Toy Story (1995.0) | bi_score=0.6955 | director=John Lasseter
4. Toy Story of Terror! (2013.0) | bi_score=0.6739 | director=Angus MacLane
5. The Pixar Story (2007.0) | bi_score=0.6688 | director=Leslie Iwerks

After re-ranking top-5:
1. Toy Story (1995.0) | bi_score=0.6955 | rerank_score=7.9102 | director=John Lasseter
2. Toy Story 2 (1999.0) | bi_score=0.7075 | rerank_score=7.8594 | director=John Lasseter
3. Toy Story 3 (2010.0) | bi_score=0.7242 | rerank_score=7.4759 | director=Lee Unkrich
4. Toy Story That Time Forgot (2014.0) | bi_score=0.6582 | rerank_score=6.9290 | director=Steve Purcell
5. Toy Story of Terror! (2013.0) | bi_score=0.6739 | rerank_score=6.7335 | director=Angus MacLane
QUERY: What is the plot of The Godfather?

Bi-encoder top-5:
1. The Godfather (1972.0) | bi_score=0.7475 | director=

In [ ]:
# CELL 12 — Load local generator LLM

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

GEN_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)

print("Loading local LLM...")
gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

generator = pipeline(
    "text-generation",
    model=gen_model,
    tokenizer=tokenizer
)

print("Generator loaded:", GEN_MODEL_NAME)

Loading tokenizer...
Loading local LLM...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Generator loaded: Qwen/Qwen2.5-1.5B-Instruct


In [ ]:
# CELL 13 — Prompt template and generation function

STRICT_SYSTEM_PROMPT = """
You are a movie question-answering assistant.

Rules:
1. Answer ONLY using the provided movie context.
2. If the context does not contain enough information to answer, say:
   "I don't have information about that in the provided context."
3. Do not use outside knowledge.
4. Cite the movie title(s) you used in the answer.
5. Keep the answer concise and factual.
"""

def build_context(reranked_results, max_docs=5):
    context_blocks = []

    for i, r in enumerate(reranked_results[:max_docs], start=1):
        block = (
            f"[Movie {i}]\n"
            f"Title: {r['title']}\n"
            f"Year: {int(r['year']) if not pd.isna(r['year']) else 'Unknown'}\n"
            f"Director: {r['director']}\n"
            f"Genres: {movies.iloc[r['faiss_id']]['genres_text']}\n"
            f"Cast: {r['cast_text']}\n"
            f"Overview: {movies.iloc[r['faiss_id']]['overview']}"
        )
        context_blocks.append(block)

    return "\n\n".join(context_blocks)


def generate_answer(query, reranked_results, max_new_tokens=180):
    context = build_context(reranked_results, max_docs=5)

    messages = [
        {
            "role": "system",
            "content": STRICT_SYSTEM_PROMPT.strip()
        },
        {
            "role": "user",
            "content": (
                f"Movie context:\n{context}\n\n"
                f"Question: {query}\n\n"
                f"Answer:"
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    output = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        return_full_text=False,
        pad_token_id=tokenizer.eos_token_id
    )

    answer = output[0]["generated_text"].strip()
    return answer

In [ ]:
# CELL 13B — Safer context builder

def build_context(reranked_results, max_docs=5):
    context_blocks = []

    for i, r in enumerate(reranked_results[:max_docs], start=1):
        year = r.get("year", "Unknown")
        year_text = int(year) if not pd.isna(year) else "Unknown"

        block = (
            f"[Movie {i}]\n"
            f"Title: {r.get('title', 'Unknown')}\n"
            f"Year: {year_text}\n"
            f"Director: {r.get('director', 'Unknown')}\n"
            f"Genres: {r.get('genres_text', 'Unknown')}\n"
            f"Cast: {r.get('cast_text', 'Unknown')}\n"
            f"Overview: {r.get('overview', '')}"
        )
        context_blocks.append(block)

    return "\n\n".join(context_blocks)

In [ ]:
# CELL 14 — End-to-end generation test

test_generation_queries = [
    "Who directed Toy Story?",
    "What is the plot of The Godfather?",
    "Who directed Inception?",
    "What is Avatar about?",
    "Who directed Oppenheimer?"
]

for query in test_generation_queries:
    print("=" * 100)
    print("QUERY:", query)

    candidates = retrieve(query, k=20)
    reranked_results = rerank(query, candidates, n=5)
    answer = generate_answer(query, reranked_results)

    print("\nRetrieved titles after re-ranking:")
    for i, r in enumerate(reranked_results, start=1):
        print(f"{i}. {r['title']} ({r['year']}) | rerank_score={r['rerank_score']:.4f}")

    print("\nAnswer:")
    print(answer)

[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUERY: Who directed Toy Story?


[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Retrieved titles after re-ranking:
1. Toy Story (1995.0) | rerank_score=7.9102
2. Toy Story 2 (1999.0) | rerank_score=7.8594
3. Toy Story 3 (2010.0) | rerank_score=7.4759
4. Toy Story That Time Forgot (2014.0) | rerank_score=6.9290
5. Toy Story of Terror! (2013.0) | rerank_score=6.7335

Answer:
John Lasseter directed Toy Story.
QUERY: What is the plot of The Godfather?


[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Retrieved titles after re-ranking:
1. The Godfather: Part III (1990.0) | rerank_score=3.0260
2. The Godfather (1972.0) | rerank_score=2.1102
3. The Godfather: Part II (1974.0) | rerank_score=2.0538
4. The Godfather Trilogy: 1972-1990 (1992.0) | rerank_score=1.5875
5. The New Godfathers (1979.0) | rerank_score=1.5033

Answer:
The Godfather spans three films from 1972 to 1990, focusing on the fictional Italian-American Corleone crime family. It begins with Vito Corleone's survival after being nearly killed by rival gangsters in 1945, leading to his eldest son, Michael, stepping in to manage the family business. The film then follows Michael as he expands the family's operations into various locations including Las Vegas, Hollywood, and Cuba. The third installment, The Godfather: Part III, continues this story, showing Michael's efforts to legitimize his business dealings in 1979 New York and Italy while seeking forgiveness for his past actions.
QUERY: Who directed Inception?


[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Retrieved titles after re-ranking:
1. Inception (2010.0) | rerank_score=6.6345
2. Transcendent Man (2009.0) | rerank_score=-3.1799
3. Directed by Sidney Lumet: How the Devil Was Made (2008.0) | rerank_score=-3.3339
4. Dream Work (2001.0) | rerank_score=-3.3984
5. Hollywood between Paranoia and Sci-Fi. The Power of Myth (2011.0) | rerank_score=-3.9083

Answer:
Christopher Nolan directed Inception.
QUERY: What is Avatar about?


[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Retrieved titles after re-ranking:
1. My Avatar and Me (2010.0) | rerank_score=3.4476
2. Avatar (2009.0) | rerank_score=0.2604
3. Avatar: Creating the World of Pandora (2010.0) | rerank_score=-0.0429
4. Avatar 2 (2020.0) | rerank_score=-0.9305
5. The Last Airbender (2010.0) | rerank_score=-4.2795

Answer:
Avatar is an action-adventure fantasy science fiction film directed by James Cameron. It tells the story of Jake Sully, a paraplegic Marine sent to the moon Pandora on a mission to save the Na'vi people from extinction at the hands of the human mining company Wrecker Industries. The film explores themes of identity, morality, and the impact of technology on society.
QUERY: Who directed Oppenheimer?

Retrieved titles after re-ranking:
1. The Day After Trinity (1981.0) | rerank_score=6.8177
2. Alchemy (2005.0) | rerank_score=5.6452
3. Day One (1989.0) | rerank_score=4.0975
4. Tar (1997.0) | rerank_score=-4.2215
5. Directed by Sidney Lumet: How the Devil Was Made (2008.0) | rerank_score

In [ ]:
# CELL 15 — Safer answer generation with retrieval confidence threshold

MIN_RERANK_SCORE = 0.0

def should_refuse(reranked_results, threshold=MIN_RERANK_SCORE):
    if not reranked_results:
        return True

    top_score = reranked_results[0].get("rerank_score", -999)

    # If the best re-ranked document is still weak, refuse.
    if top_score < threshold:
        return True

    return False


def generate_answer_safe(query, reranked_results, max_new_tokens=180):
    if should_refuse(reranked_results):
        return "I don't have information about that in the provided context."

    context = build_context(reranked_results, max_docs=5)

    messages = [
        {
            "role": "system",
            "content": STRICT_SYSTEM_PROMPT.strip()
        },
        {
            "role": "user",
            "content": (
                f"Movie context:\n{context}\n\n"
                f"Question: {query}\n\n"
                f"Answer with citation using movie title(s):"
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    output = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,
        pad_token_id=tokenizer.eos_token_id
    )

    answer = output[0]["generated_text"].strip()
    return answer

In [ ]:
# CELL 16 — Re-test safer generation

for query in test_generation_queries:
    print("=" * 100)
    print("QUERY:", query)

    candidates = retrieve(query, k=20)
    reranked_results = rerank(query, candidates, n=5)
    answer = generate_answer_safe(query, reranked_results)

    print("\nRetrieved titles after re-ranking:")
    for i, r in enumerate(reranked_results, start=1):
        print(f"{i}. {r['title']} ({r['year']}) | rerank_score={r['rerank_score']:.4f}")

    print("\nAnswer:")
    print(answer)

[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUERY: Who directed Toy Story?


[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Retrieved titles after re-ranking:
1. Toy Story (1995.0) | rerank_score=7.9102
2. Toy Story 2 (1999.0) | rerank_score=7.8594
3. Toy Story 3 (2010.0) | rerank_score=7.4759
4. Toy Story That Time Forgot (2014.0) | rerank_score=6.9290
5. Toy Story of Terror! (2013.0) | rerank_score=6.7335

Answer:
Toy Story was directed by John Lasseter.
QUERY: What is the plot of The Godfather?


[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Retrieved titles after re-ranking:
1. The Godfather: Part III (1990.0) | rerank_score=3.0260
2. The Godfather (1972.0) | rerank_score=2.1102
3. The Godfather: Part II (1974.0) | rerank_score=2.0538
4. The Godfather Trilogy: 1972-1990 (1992.0) | rerank_score=1.5875
5. The New Godfathers (1979.0) | rerank_score=1.5033

Answer:
The Godfather (1972)
Spanning the years 1945 to 1955, a chronicle of the fictional Italian-American Corleone crime family. When organized crime family patriarch, Vito Corleone barely survives an attempt on his life, his youngest son, Michael steps in to take care of the would-be killers, launching a campaign of bloody revenge.
QUERY: Who directed Inception?


[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Retrieved titles after re-ranking:
1. Inception (2010.0) | rerank_score=6.6345
2. Transcendent Man (2009.0) | rerank_score=-3.1799
3. Directed by Sidney Lumet: How the Devil Was Made (2008.0) | rerank_score=-3.3339
4. Dream Work (2001.0) | rerank_score=-3.3984
5. Hollywood between Paranoia and Sci-Fi. The Power of Myth (2011.0) | rerank_score=-3.9083

Answer:
Christopher Nolan directed Inception.
QUERY: What is Avatar about?


[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Retrieved titles after re-ranking:
1. My Avatar and Me (2010.0) | rerank_score=3.4476
2. Avatar (2009.0) | rerank_score=0.2604
3. Avatar: Creating the World of Pandora (2010.0) | rerank_score=-0.0429
4. Avatar 2 (2020.0) | rerank_score=-0.9305
5. The Last Airbender (2010.0) | rerank_score=-4.2795

Answer:
Avatar is an action-adventure fantasy science fiction film directed by James Cameron. It tells the story of Jake Sully, a paralyzed U.S. Marine sent to the planet Pandora to fight for the peacekeeping forces against the Wrecks, while also exploring the culture and ecology of the Na'vi people. The film features special effects created by James Cameron himself and was inspired by the concept of Avatar, a fictional underwater society created by author James Cameron.
QUERY: Who directed Oppenheimer?

Retrieved titles after re-ranking:
1. The Day After Trinity (1981.0) | rerank_score=6.8177
2. Alchemy (2005.0) | rerank_score=5.6452
3. Day One (1989.0) | rerank_score=4.0975
4. Tar (1997.0)

In [ ]:
# CELL 17 — Title-aware re-ranking fix for sequel/title confusion

import re

# Lowercase title lookup for exact title matching
title_to_indices = {}
for idx, title in enumerate(movies["title"].astype(str)):
    title_to_indices.setdefault(title.lower(), []).append(idx)

# Sort titles by length descending so "The Godfather: Part II" is checked before "The Godfather"
all_titles_lower = sorted(title_to_indices.keys(), key=len, reverse=True)


def normalize_text_for_match(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s:']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def find_exact_title_in_query(query):
    q = normalize_text_for_match(query)

    for title_lower in all_titles_lower:
        title_norm = normalize_text_for_match(title_lower)

        # Avoid matching very short generic titles accidentally
        if len(title_norm) < 4:
            continue

        pattern = r"\b" + re.escape(title_norm) + r"\b"
        if re.search(pattern, q):
            return title_lower

    return None


def rerank_title_aware(query, candidates, n=5, title_boost=5.0):
    reranked = rerank(query, candidates, n=len(candidates))

    matched_title = find_exact_title_in_query(query)

    if matched_title is not None:
        for item in reranked:
            item_title_lower = str(item["title"]).lower()

            # Strong boost only for exact title match
            if item_title_lower == matched_title:
                item["rerank_score"] += title_boost
                item["title_match_boost"] = title_boost
            else:
                item["title_match_boost"] = 0.0

    reranked = sorted(
        reranked,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return reranked[:n]

In [ ]:
# CELL 17B — Safer title-aware matching to avoid Alien vs Alien³ bug

import re
from collections import defaultdict

def normalize_text_for_match(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s:']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Build normalized title records
title_records = []
for idx, row in movies.iterrows():
    raw_title = str(row["title"])
    norm_title = normalize_text_for_match(raw_title)

    if len(norm_title) >= 4:
        title_records.append({
            "idx": idx,
            "title": raw_title,
            "title_lower": raw_title.lower(),
            "norm_title": norm_title,
            "year": row["release_year"]
        })

# Group by normalized title
norm_to_records = defaultdict(list)
for rec in title_records:
    norm_to_records[rec["norm_title"]].append(rec)

def choose_canonical_record(records):
    """
    If multiple titles normalize to same string, prefer the clean exact title.
    Example: Alien should beat Alien³ because Alien.lower() == normalized title.
    Then prefer shorter title, then earlier year.
    """
    def sort_key(rec):
        exact_clean = 0 if rec["title_lower"] == rec["norm_title"] else 1
        title_len = len(rec["title"])
        year = rec["year"] if not pd.isna(rec["year"]) else 9999
        return (exact_clean, title_len, year)

    return sorted(records, key=sort_key)[0]

canonical_title_records = [
    choose_canonical_record(records)
    for records in norm_to_records.values()
]

# Longest title first
canonical_title_records = sorted(
    canonical_title_records,
    key=lambda r: len(r["norm_title"]),
    reverse=True
)

def find_exact_title_record_in_query(query):
    q = normalize_text_for_match(query)

    for rec in canonical_title_records:
        title_norm = rec["norm_title"]
        pattern = r"\b" + re.escape(title_norm) + r"\b"

        if re.search(pattern, q):
            return rec

    return None

def find_exact_title_in_query(query):
    rec = find_exact_title_record_in_query(query)
    return None if rec is None else rec["title"].lower()


def rerank_title_aware(query, candidates, n=5, title_boost=5.0):
    reranked = rerank(query, candidates, n=len(candidates))

    matched_rec = find_exact_title_record_in_query(query)

    if matched_rec is not None:
        matched_title = matched_rec["title"]

        for item in reranked:
            if str(item["title"]) == matched_title:
                item["rerank_score"] += title_boost
                item["title_match_boost"] = title_boost
            else:
                item["title_match_boost"] = 0.0
    else:
        for item in reranked:
            item["title_match_boost"] = 0.0

    reranked = sorted(
        reranked,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return reranked[:n]

In [ ]:
# CELL 18 — Compare normal rerank vs title-aware rerank

title_fix_queries = [
    "Who directed Toy Story?",
    "What is the plot of The Godfather?",
    "Who directed Alien?",
    "What is Avatar about?",
    "Who directed Oppenheimer?"
]

for query in title_fix_queries:
    print("=" * 100)
    print("QUERY:", query)
    print("Matched title:", find_exact_title_in_query(query))

    candidates = retrieve(query, k=20)

    normal = rerank(query, candidates, n=5)
    fixed = rerank_title_aware(query, candidates, n=5)

    print("\nNormal re-ranker top-5:")
    for i, r in enumerate(normal, start=1):
        print(f"{i}. {r['title']} ({r['year']}) | score={r['rerank_score']:.4f}")

    print("\nTitle-aware re-ranker top-5:")
    for i, r in enumerate(fixed, start=1):
        boost = r.get("title_match_boost", 0.0)
        print(f"{i}. {r['title']} ({r['year']}) | score={r['rerank_score']:.4f} | boost={boost}")

QUERY: Who directed Toy Story?
Matched title: toy story

Normal re-ranker top-5:
1. Toy Story (1995.0) | score=7.9102
2. Toy Story 2 (1999.0) | score=7.8594
3. Toy Story 3 (2010.0) | score=7.4759
4. Toy Story That Time Forgot (2014.0) | score=6.9290
5. Toy Story of Terror! (2013.0) | score=6.7335

Title-aware re-ranker top-5:
1. Toy Story (1995.0) | score=12.9102 | boost=5.0
2. Toy Story 2 (1999.0) | score=7.8594 | boost=0.0
3. Toy Story 3 (2010.0) | score=7.4759 | boost=0.0
4. Toy Story That Time Forgot (2014.0) | score=6.9290 | boost=0.0
5. Toy Story of Terror! (2013.0) | score=6.7335 | boost=0.0
QUERY: What is the plot of The Godfather?
Matched title: the godfather

Normal re-ranker top-5:
1. The Godfather: Part III (1990.0) | score=3.0260
2. The Godfather (1972.0) | score=2.1102
3. The Godfather: Part II (1974.0) | score=2.0538
4. The Godfather Trilogy: 1972-1990 (1992.0) | score=1.5875
5. The New Godfathers (1979.0) | score=1.5033

Title-aware re-ranker top-5:
1. The Godfather (19

In [ ]:
# CELL 19 — Explicit-title unanswerable guard

def extract_candidate_title_from_query(query):
    """
    Simple heuristic for explicit movie-title questions.
    Used only for refusal guard, not for retrieval itself.
    """
    q = str(query).strip()

    patterns = [
        r"who directed\s+(.+?)\??$",
        r"what is\s+(.+?)\s+about\??$",
        r"what is the plot of\s+(.+?)\??$",
        r"tell me about\s+(.+?)\??$",
        r"describe\s+(.+?)\??$",
    ]

    q_lower = q.lower()

    for pattern in patterns:
        match = re.search(pattern, q_lower, flags=re.IGNORECASE)
        if match:
            candidate = match.group(1).strip()
            candidate = candidate.strip(" .'\"")
            return candidate

    return None


def is_explicit_title_missing(query):
    """
    If query appears to ask about a specific title,
    but no exact title exists in corpus, refuse.
    """
    candidate = extract_candidate_title_from_query(query)

    if candidate is None:
        return False

    matched_record = find_exact_title_record_in_query(query)

    if matched_record is None:
        return True

    return False


# Quick checks
for q in [
    "Who directed Toy Story?",
    "Who directed Oppenheimer?",
    "What is the plot of The Godfather?",
    "What is Avatar about?",
]:
    print(q, "=> candidate:", extract_candidate_title_from_query(q), "| missing:", is_explicit_title_missing(q))

Who directed Toy Story? => candidate: toy story | missing: False
Who directed Oppenheimer? => candidate: oppenheimer | missing: True
What is the plot of The Godfather? => candidate: the godfather | missing: False
What is Avatar about? => candidate: avatar | missing: False


In [ ]:
# CELL 20 — Final pipeline entrypoint: run_query(query: str) -> dict

def run_query(query: str) -> dict:
    """
    Public entrypoint for the QA pipeline.
    Returns:
        {
            "answer": str,
            "retrieved_titles": list[str]
        }
    """
    try:
        if query is None:
            query = ""

        query = str(query).strip()

        if len(query) == 0:
            return {
                "answer": "I don't have information about that in the provided context.",
                "retrieved_titles": []
            }

        candidates = retrieve(query, k=20)
        reranked_results = rerank_title_aware(query, candidates, n=5)

        retrieved_titles = [str(r["title"]) for r in reranked_results]

        # Explicit title not found in corpus, e.g. Oppenheimer
        if is_explicit_title_missing(query):
            return {
                "answer": "I don't have information about that in the provided context.",
                "retrieved_titles": retrieved_titles
            }

        answer = generate_answer_safe(query, reranked_results)

        return {
            "answer": str(answer),
            "retrieved_titles": retrieved_titles
        }

    except Exception as e:
        return {
            "answer": "I don't have information about that in the provided context.",
            "retrieved_titles": []
        }

In [ ]:
# CELL 21 — Self-test for run_query

test_inputs = [
    "Who directed Inception?",
    "asdfgh nonsense query",
    "",
    "Bir Zamanlar Anadolu'da hakkında ne biliyorsun?",
    "Who directed Oppenheimer?",
    "Who directed Alien?",
    "What is Avatar about?"
]

for q in test_inputs:
    r = run_query(q)

    assert isinstance(r, dict), f"must return dict, got {type(r)}"
    assert "answer" in r and "retrieved_titles" in r, f"missing keys: {list(r.keys())}"
    assert isinstance(r["answer"], str), f"answer must be str"
    assert isinstance(r["retrieved_titles"], list), f"retrieved_titles must be list"
    assert all(isinstance(t, str) for t in r["retrieved_titles"]), "all titles must be strings"

    print("=" * 100)
    print(f"QUERY: {q!r}")
    print("Retrieved titles:", r["retrieved_titles"])
    print("Answer:", r["answer"][:500])

[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUERY: 'Who directed Inception?'
Retrieved titles: ['Inception', 'Transcendent Man', 'Directed by Sidney Lumet: How the Devil Was Made', 'Dream Work', 'Hollywood between Paranoia and Sci-Fi. The Power of Myth']
Answer: Christopher Nolan directed Inception.
QUERY: 'asdfgh nonsense query'
Retrieved titles: ['Les Patterson Saves the World', 'Monster Ark', 'Radiopiratene', 'Doug Stanhope: No Refunds', 'Myq Kaplan: Small, Dork and Handsome']
Answer: I don't have information about that in the provided context.
QUERY: ''
Retrieved titles: []
Answer: I don't have information about that in the provided context.
QUERY: "Bir Zamanlar Anadolu'da hakkında ne biliyorsun?"
Retrieved titles: ['Anatolian Eagles', 'Kolpaçino', 'Hokkabaz', 'Ya Sonra?', 'Dabbe: Bir cin vakasi']
Answer: I don't have information about that in the provided context.
QUERY: 'Who directed Oppenheimer?'
Retrieved titles: ['The Day After Trinity', 'Alchemy', 'Day One', 'Tar', 'Directed by Sidney Lumet: How the Devil Was Made']
An

[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUERY: 'Who directed Alien?'
Retrieved titles: ['Alien', 'Alien 2: On Earth', 'The Alien Saga', 'Alien Origin', 'Alien Apocalypse']
Answer: Ridley Scott directed Alien.


[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUERY: 'What is Avatar about?'
Retrieved titles: ['Avatar', 'My Avatar and Me', 'Avatar: Creating the World of Pandora', 'Avatar 2', 'The Last Airbender']
Answer: Avatar is an action-adventure science fiction film directed by James Cameron. It tells the story of Jake Sully, a paraplegic Marine sent to the moon Pandora on a mission to protect an indigenous alien civilization from exploitation by humans.


In [ ]:
# CELL 22 — Build 20-query test set

test_set = [
    # Standard queries
    {
        "category": "standard",
        "query": "Who directed Inception?",
        "gold_titles": ["Inception"]
    },
    {
        "category": "standard",
        "query": "Who directed The Matrix?",
        "gold_titles": ["The Matrix"]
    },
    {
        "category": "standard",
        "query": "What is Avatar about?",
        "gold_titles": ["Avatar"]
    },
    {
        "category": "standard",
        "query": "Who directed Titanic?",
        "gold_titles": ["Titanic"]
    },
    {
        "category": "standard",
        "query": "What is Finding Nemo about?",
        "gold_titles": ["Finding Nemo"]
    },

    # Adversarial queries: shared vocabulary across many movies
    {
        "category": "adversarial",
        "query": "A movie about dreams inside dreams and stealing secrets from the mind",
        "gold_titles": ["Inception"]
    },
    {
        "category": "adversarial",
        "query": "A movie about a computer hacker discovering reality is simulated",
        "gold_titles": ["The Matrix"]
    },
    {
        "category": "adversarial",
        "query": "A movie about toys that come alive when humans are not around",
        "gold_titles": ["Toy Story"]
    },
    {
        "category": "adversarial",
        "query": "A movie about a clownfish father searching for his son",
        "gold_titles": ["Finding Nemo"]
    },
    {
        "category": "adversarial",
        "query": "A movie about a ship disaster and a romance across social classes",
        "gold_titles": ["Titanic"]
    },

    # Unanswerable queries: post-2017 / not in corpus
    {
        "category": "unanswerable",
        "query": "Who directed Oppenheimer?",
        "gold_titles": []
    },
    {
        "category": "unanswerable",
        "query": "What is Dune 2021 about?",
        "gold_titles": []
    },
    {
        "category": "unanswerable",
        "query": "Who directed Everything Everywhere All at Once?",
        "gold_titles": []
    },
    {
        "category": "unanswerable",
        "query": "What is The Completely Fictional Banana Spaceship Movie about?",
        "gold_titles": []
    },

    # Sequel-confusion queries
    {
        "category": "sequel_confusion",
        "query": "Who directed Toy Story?",
        "gold_titles": ["Toy Story"]
    },
    {
        "category": "sequel_confusion",
        "query": "What is the plot of The Godfather?",
        "gold_titles": ["The Godfather"]
    },
    {
        "category": "sequel_confusion",
        "query": "Who directed Alien?",
        "gold_titles": ["Alien"]
    },
    {
        "category": "sequel_confusion",
        "query": "What is the plot of Terminator?",
        "gold_titles": ["The Terminator"]
    },

    # Extra mixed queries
    {
        "category": "standard",
        "query": "Who directed Pulp Fiction?",
        "gold_titles": ["Pulp Fiction"]
    },
    {
        "category": "standard",
        "query": "What is The Dark Knight about?",
        "gold_titles": ["The Dark Knight"]
    },
]

test_df = pd.DataFrame(test_set)
display(test_df)

print("Number of test queries:", len(test_df))
print(test_df["category"].value_counts())

,category,query,gold_titles
0,standard,Who directed Inception?,[Inception]
1,standard,Who directed The Matrix?,[The Matrix]
2,standard,What is Avatar about?,[Avatar]
3,standard,Who directed Titanic?,[Titanic]
4,standard,What is Finding Nemo about?,[Finding Nemo]
5,adversarial,A movie about dreams inside dreams and stealin...,[Inception]
6,adversarial,A movie about a computer hacker discovering re...,[The Matrix]
7,adversarial,A movie about toys that come alive when humans...,[Toy Story]
8,adversarial,A movie about a clownfish father searching for...,[Finding Nemo]
9,adversarial,A movie about a ship disaster and a romance ac...,[Titanic]


Number of test queries: 20
category
standard            7
adversarial         5
unanswerable        4
sequel_confusion    4
Name: count, dtype: int64


In [ ]:
# CELL 23 — Retrieval metric helper functions

def hit_at_k(retrieved_titles, gold_titles, k):
    if len(gold_titles) == 0:
        return None

    retrieved_at_k = retrieved_titles[:k]

    for gold in gold_titles:
        if gold in retrieved_at_k:
            return 1

    return 0


def reciprocal_rank(retrieved_titles, gold_titles):
    if len(gold_titles) == 0:
        return None

    for i, title in enumerate(retrieved_titles, start=1):
        if title in gold_titles:
            return 1.0 / i

    return 0.0


def evaluate_retrieval(test_df, mode="biencoder", k_retrieve=20, n_final=5):
    rows = []

    for _, row in test_df.iterrows():
        query = row["query"]
        gold_titles = row["gold_titles"]

        candidates = retrieve(query, k=k_retrieve)

        if mode == "biencoder":
            final_results = candidates[:max(10, n_final)]
        elif mode == "reranker":
            final_results = rerank(query, candidates, n=max(10, n_final))
        elif mode == "title_aware_reranker":
            final_results = rerank_title_aware(query, candidates, n=max(10, n_final))
        else:
            raise ValueError("Unknown mode")

        retrieved_titles = [r["title"] for r in final_results]

        rows.append({
            "category": row["category"],
            "query": query,
            "gold_titles": gold_titles,
            "retrieved_titles": retrieved_titles[:10],
            "R@5": hit_at_k(retrieved_titles, gold_titles, 5),
            "R@10": hit_at_k(retrieved_titles, gold_titles, 10),
            "MRR": reciprocal_rank(retrieved_titles, gold_titles)
        })

    result_df = pd.DataFrame(rows)

    answerable_df = result_df[result_df["gold_titles"].apply(lambda x: len(x) > 0)].copy()

    metrics = {
        "mode": mode,
        "Recall@5": answerable_df["R@5"].mean(),
        "Recall@10": answerable_df["R@10"].mean(),
        "MRR": answerable_df["MRR"].mean(),
        "num_answerable_queries": len(answerable_df)
    }

    return result_df, metrics

In [ ]:
# CELL 24 — Evaluate retrieval modes

biencoder_df, biencoder_metrics = evaluate_retrieval(
    test_df,
    mode="biencoder",
    k_retrieve=20,
    n_final=5
)

reranker_df, reranker_metrics = evaluate_retrieval(
    test_df,
    mode="reranker",
    k_retrieve=20,
    n_final=5
)

title_aware_df, title_aware_metrics = evaluate_retrieval(
    test_df,
    mode="title_aware_reranker",
    k_retrieve=20,
    n_final=5
)

metrics_df = pd.DataFrame([
    biencoder_metrics,
    reranker_metrics,
    title_aware_metrics
])

display(metrics_df)

print("\nTitle-aware detailed retrieval results:")
display(title_aware_df[[
    "category", "query", "gold_titles", "retrieved_titles", "R@5", "R@10", "MRR"
]])

,mode,Recall@5,Recall@10,MRR,num_answerable_queries
0,biencoder,0.6875,0.75,0.591146,16
1,reranker,0.7500,0.75,0.528125,16
2,title_aware_reranker,0.7500,0.75,0.718750,16



Title-aware detailed retrieval results:


,category,query,gold_titles,retrieved_titles,R@5,R@10,MRR
0,standard,Who directed Inception?,[Inception],"[Inception, Transcendent Man, Directed by Sidn...",1.0,1.0,1.0
1,standard,Who directed The Matrix?,[The Matrix],"[The Matrix, The Matrix Revisited, Return to S...",1.0,1.0,1.0
2,standard,What is Avatar about?,[Avatar],"[Avatar, My Avatar and Me, Avatar: Creating th...",1.0,1.0,1.0
3,standard,Who directed Titanic?,[Titanic],"[Titanic, Titanic 2, Raise the Titanic, Titani...",1.0,1.0,1.0
4,standard,What is Finding Nemo about?,[Finding Nemo],"[Finding Nemo, Finding Dory, Captain Nemo and ...",1.0,1.0,1.0
5,adversarial,A movie about dreams inside dreams and stealin...,[Inception],"[Dreamscape, Passion of Mind, Meshes of the Af...",0.0,0.0,0.0
6,adversarial,A movie about a computer hacker discovering re...,[The Matrix],"[The Zero Theorem, War Games: The Dead Code, A...",0.0,0.0,0.0
7,adversarial,A movie about toys that come alive when humans...,[Toy Story],"[Silent Night, Deadly Night 5: The Toy Maker, ...",0.0,0.0,0.0
8,adversarial,A movie about a clownfish father searching for...,[Finding Nemo],"[Finding Nemo, Fathers' Day, Our Father, Fathe...",1.0,1.0,1.0
9,adversarial,A movie about a ship disaster and a romance ac...,[Titanic],"[Sea Wife, Message in a Bottle, Stormy Waters,...",0.0,0.0,0.0


In [ ]:
# CELL 25 — Run final pipeline on the full test set

end_to_end_rows = []

for _, row in test_df.iterrows():
    query = row["query"]
    result = run_query(query)

    end_to_end_rows.append({
        "category": row["category"],
        "query": query,
        "gold_titles": row["gold_titles"],
        "retrieved_titles": result["retrieved_titles"],
        "answer": result["answer"]
    })

end_to_end_df = pd.DataFrame(end_to_end_rows)

display(end_to_end_df)

# Save for report / later RAGAS
RESULTS_DIR = Path("/content/drive/MyDrive/movie_rag_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

end_to_end_df.to_csv(RESULTS_DIR / "end_to_end_outputs.csv", index=False)
metrics_df.to_csv(RESULTS_DIR / "retrieval_metrics_summary.csv", index=False)
title_aware_df.to_csv(RESULTS_DIR / "title_aware_retrieval_details.csv", index=False)

print("Saved outputs to:", RESULTS_DIR)

[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

,category,query,gold_titles,retrieved_titles,answer
0,standard,Who directed Inception?,[Inception],"[Inception, Transcendent Man, Directed by Sidn...",Christopher Nolan directed Inception.
1,standard,Who directed The Matrix?,[The Matrix],"[The Matrix, The Matrix Revisited, Return to S...",The Matrix was directed by Lana Wachowski.
2,standard,What is Avatar about?,[Avatar],"[Avatar, My Avatar and Me, Avatar: Creating th...",Avatar is an action-adventure science fiction ...
3,standard,Who directed Titanic?,[Titanic],"[Titanic, Titanic 2, Raise the Titanic, Titani...",James Cameron directed Titanic.
4,standard,What is Finding Nemo about?,[Finding Nemo],"[Finding Nemo, Finding Dory, Captain Nemo and ...",Finding Nemo is about a young clownfish named ...
5,adversarial,A movie about dreams inside dreams and stealin...,[Inception],"[Dreamscape, Passion of Mind, Meshes of the Af...",I don't have information about that in the pro...
6,adversarial,A movie about a computer hacker discovering re...,[The Matrix],"[The Zero Theorem, War Games: The Dead Code, A...",I don't have information about that in the pro...
7,adversarial,A movie about toys that come alive when humans...,[Toy Story],"[Silent Night, Deadly Night 5: The Toy Maker, ...",I don't have information about that in the pro...
8,adversarial,A movie about a clownfish father searching for...,[Finding Nemo],"[Finding Nemo, Fathers' Day, Our Father, Fathe...",Finding Nemo
9,adversarial,A movie about a ship disaster and a romance ac...,[Titanic],"[Sea Wife, Message in a Bottle, Stormy Waters,...",I don't have information about that in the pro...


Saved outputs to: /content/drive/MyDrive/movie_rag_results


In [ ]:
# CELL 26 — Ablation B: top-k sweep for re-ranker

def evaluate_topk_sweep(test_df, k_values=[5, 10, 20], n_final=5):
    rows = []

    for k in k_values:
        query_rows = []
        latencies = []

        for _, row in test_df.iterrows():
            query = row["query"]
            gold_titles = row["gold_titles"]

            start = time.perf_counter()

            candidates = retrieve(query, k=k)
            final_results = rerank_title_aware(query, candidates, n=n_final)

            latency_ms = (time.perf_counter() - start) * 1000
            latencies.append(latency_ms)

            retrieved_titles = [r["title"] for r in final_results]

            query_rows.append({
                "query": query,
                "category": row["category"],
                "gold_titles": gold_titles,
                "retrieved_titles": retrieved_titles,
                "R@5": hit_at_k(retrieved_titles, gold_titles, 5),
                "MRR": reciprocal_rank(retrieved_titles, gold_titles),
                "latency_ms": latency_ms
            })

        detail_df = pd.DataFrame(query_rows)
        answerable_df = detail_df[detail_df["gold_titles"].apply(lambda x: len(x) > 0)].copy()

        rows.append({
            "top_k_before_rerank": k,
            "R@5": answerable_df["R@5"].mean(),
            "MRR": answerable_df["MRR"].mean(),
            "mean_latency_ms": np.mean(latencies),
            "median_latency_ms": np.median(latencies),
            "num_queries": len(detail_df)
        })

    return pd.DataFrame(rows)

ablation_b_df = evaluate_topk_sweep(test_df, k_values=[5, 10, 20], n_final=5)

display(ablation_b_df)

ablation_b_df.to_csv(RESULTS_DIR / "ablation_b_topk_sweep.csv", index=False)

,top_k_before_rerank,R@5,MRR,mean_latency_ms,median_latency_ms,num_queries
0,5,0.6875,0.65625,1464.819337,1368.428313,20
1,10,0.7500,0.71875,1484.658916,1382.027053,20
2,20,0.7500,0.71875,1660.506365,1422.386615,20


In [ ]:
# CELL 27 — Prompt variants for Ablation C

PERMISSIVE_SYSTEM_PROMPT = """
Use the provided movie context to answer the user's question.
"""

STRICT_SYSTEM_PROMPT_ABLATION = """
You are a grounded movie question-answering assistant.

Rules:
1. Answer ONLY using the provided movie context.
2. If the provided context does not contain the answer, say exactly:
   "I don't have information about that in the provided context."
3. Do NOT use outside knowledge.
4. Do NOT guess.
5. Cite the movie title(s) used in your answer.
6. Keep the answer concise and factual.
"""

def generate_answer_with_prompt(query, reranked_results, system_prompt, max_new_tokens=180):
    context = build_context(reranked_results, max_docs=5)

    messages = [
        {
            "role": "system",
            "content": system_prompt.strip()
        },
        {
            "role": "user",
            "content": (
                f"Movie context:\n{context}\n\n"
                f"Question: {query}\n\n"
                f"Answer:"
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    output = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,
        pad_token_id=tokenizer.eos_token_id
    )

    return output[0]["generated_text"].strip()

In [ ]:
# CELL 28 — Ablation C: permissive vs strict prompt outputs

def run_query_with_prompt_variant(query, system_prompt, use_explicit_missing_guard=False):
    try:
        query = str(query).strip()

        if len(query) == 0:
            return {
                "answer": "I don't have information about that in the provided context.",
                "retrieved_titles": []
            }

        candidates = retrieve(query, k=20)
        reranked_results = rerank_title_aware(query, candidates, n=5)
        retrieved_titles = [str(r["title"]) for r in reranked_results]

        if use_explicit_missing_guard and is_explicit_title_missing(query):
            return {
                "answer": "I don't have information about that in the provided context.",
                "retrieved_titles": retrieved_titles
            }

        answer = generate_answer_with_prompt(
            query=query,
            reranked_results=reranked_results,
            system_prompt=system_prompt
        )

        return {
            "answer": answer,
            "retrieved_titles": retrieved_titles
        }

    except Exception:
        return {
            "answer": "I don't have information about that in the provided context.",
            "retrieved_titles": []
        }


ablation_c_rows = []

for _, row in test_df.iterrows():
    query = row["query"]

    permissive_result = run_query_with_prompt_variant(
        query,
        PERMISSIVE_SYSTEM_PROMPT,
        use_explicit_missing_guard=False
    )

    strict_result = run_query_with_prompt_variant(
        query,
        STRICT_SYSTEM_PROMPT_ABLATION,
        use_explicit_missing_guard=True
    )

    ablation_c_rows.append({
        "category": row["category"],
        "query": query,
        "gold_titles": row["gold_titles"],
        "permissive_answer": permissive_result["answer"],
        "strict_answer": strict_result["answer"],
        "permissive_retrieved_titles": permissive_result["retrieved_titles"],
        "strict_retrieved_titles": strict_result["retrieved_titles"]
    })

ablation_c_df = pd.DataFrame(ablation_c_rows)
display(ablation_c_df)

ablation_c_df.to_csv(RESULTS_DIR / "ablation_c_prompt_outputs.csv", index=False)

[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

,category,query,gold_titles,permissive_answer,strict_answer,permissive_retrieved_titles,strict_retrieved_titles
0,standard,Who directed Inception?,[Inception],Christopher Nolan,Christopher Nolan directed Inception.,"[Inception, Transcendent Man, Directed by Sidn...","[Inception, Transcendent Man, Directed by Sidn..."
1,standard,Who directed The Matrix?,[The Matrix],Lana Wachowski directed The Matrix.,The Matrix was directed by Lana Wachowski.,"[The Matrix, The Matrix Revisited, Return to S...","[The Matrix, The Matrix Revisited, Return to S..."
2,standard,What is Avatar about?,[Avatar],Avatar is an action-adventure science fiction ...,Avatar is an action-adventure science fiction ...,"[Avatar, My Avatar and Me, Avatar: Creating th...","[Avatar, My Avatar and Me, Avatar: Creating th..."
3,standard,Who directed Titanic?,[Titanic],James Cameron directed Titanic.,James Cameron directed Titanic.,"[Titanic, Titanic 2, Raise the Titanic, Titani...","[Titanic, Titanic 2, Raise the Titanic, Titani..."
4,standard,What is Finding Nemo about?,[Finding Nemo],Finding Nemo is an animated adventure film set...,Finding Nemo is an animated adventure film set...,"[Finding Nemo, Finding Dory, Captain Nemo and ...","[Finding Nemo, Finding Dory, Captain Nemo and ..."
5,adversarial,A movie about dreams inside dreams and stealin...,[Inception],Dreamscape,Dreamscape,"[Dreamscape, Passion of Mind, Meshes of the Af...","[Dreamscape, Passion of Mind, Meshes of the Af..."
6,adversarial,A movie about a computer hacker discovering re...,[The Matrix],"The movie you are looking for is ""War Games: T...",The Zero Theorem,"[The Zero Theorem, War Games: The Dead Code, A...","[The Zero Theorem, War Games: The Dead Code, A..."
7,adversarial,A movie about toys that come alive when humans...,[Toy Story],The Human Duplicators,The Human Duplicators,"[Silent Night, Deadly Night 5: The Toy Maker, ...","[Silent Night, Deadly Night 5: The Toy Maker, ..."
8,adversarial,A movie about a clownfish father searching for...,[Finding Nemo],Finding Nemo,Finding Nemo,"[Finding Nemo, Fathers' Day, Our Father, Fathe...","[Finding Nemo, Fathers' Day, Our Father, Fathe..."
9,adversarial,A movie about a ship disaster and a romance ac...,[Titanic],"The movie that fits this description is ""Sea W...",The movie that fits this description is **Sea ...,"[Sea Wife, Message in a Bottle, Stormy Waters,...","[Sea Wife, Message in a Bottle, Stormy Waters,..."


In [ ]:
# CELL 29 — Hallucination count for unanswerable queries

REFUSAL_TEXT = "I don't have information about that in the provided context."

unanswerable_prompt_df = ablation_c_df[
    ablation_c_df["category"] == "unanswerable"
].copy()

def is_refusal(answer):
    return REFUSAL_TEXT.lower() in str(answer).lower()

permissive_refusals = unanswerable_prompt_df["permissive_answer"].apply(is_refusal).sum()
strict_refusals = unanswerable_prompt_df["strict_answer"].apply(is_refusal).sum()

num_unanswerable = len(unanswerable_prompt_df)

ablation_c_summary = pd.DataFrame([
    {
        "prompt_type": "permissive",
        "num_unanswerable": num_unanswerable,
        "num_refusals": permissive_refusals,
        "num_hallucinated_answers": num_unanswerable - permissive_refusals
    },
    {
        "prompt_type": "strict_plus_missing_title_guard",
        "num_unanswerable": num_unanswerable,
        "num_refusals": strict_refusals,
        "num_hallucinated_answers": num_unanswerable - strict_refusals
    }
])

display(ablation_c_summary)

print("\nUnanswerable query outputs:")
display(unanswerable_prompt_df[[
    "query",
    "permissive_answer",
    "strict_answer",
    "permissive_retrieved_titles",
    "strict_retrieved_titles"
]])

ablation_c_summary.to_csv(RESULTS_DIR / "ablation_c_summary.csv", index=False)

,prompt_type,num_unanswerable,num_refusals,num_hallucinated_answers
0,permissive,4,0,4
1,strict_plus_missing_title_guard,4,3,1



Unanswerable query outputs:


,query,permissive_answer,strict_answer,permissive_retrieved_titles,strict_retrieved_titles
10,Who directed Oppenheimer?,Jon Else,I don't have information about that in the pro...,"[The Day After Trinity, Alchemy, Day One, Tar,...","[The Day After Trinity, Alchemy, Day One, Tar,..."
11,What is Dune 2021 about?,Dune 2021 is not a widely recognized term or t...,I don't have information about that in the pro...,"[Dune, Frank Herbert's Dune, Jodorowsky's Dune...","[Dune, Frank Herbert's Dune, Jodorowsky's Dune..."
12,Who directed Everything Everywhere All at Once?,Dan Rush,Everything Everywhere All at Once was directed...,"[Everything, Everything Put Together, Everythi...","[Everything, Everything Put Together, Everythi..."
13,What is The Completely Fictional Banana Spaces...,The completely fictional Banana Spaceship Movi...,I don't have information about that in the pro...,"[Banana, Bananas!*, Banana Paradise, Pacific B...","[Banana, Bananas!*, Banana Paradise, Pacific B..."


In [2]:
# CELL — Ablation C: Faithfulness per prompt (RAGAS proxy via custom metric)
#
# Ablation C calls for a RAGAS-style faithfulness score for each prompt.
# Reloading the 7B judge on T4 requires another runtime restart, so we use the
# same custom keyword-based faithfulness defined in CELL RAGAS-6 as a proxy.
# Logic is parallel to RAGAS faithfulness: informative tokens in the answer
# must appear somewhere in the retrieved contexts.

import re, ast
import pandas as pd
import numpy as np
from pathlib import Path

RESULTS_DIR = Path("/content/drive/MyDrive/movie_rag_results")

# Reload ablation C outputs and contexts (safe across runtime restarts)
ablation_c_df = pd.read_csv(RESULTS_DIR / "ablation_c_prompt_outputs.csv")
ragas_eval_df = pd.read_pickle(RESULTS_DIR / "ragas_eval_df.pkl")

# Align contexts to the same query order
contexts_by_query = dict(zip(ragas_eval_df["question"], ragas_eval_df["contexts"]))

STOP = set("""
a an the and or but if then so of to in on at for from with by as is are was were be been being
this that these those it its his her their there here what which who whom whose how why when
about against between into through during before after above below up down out off over under
i you he she we they me him us them my your its our ought would could should may might can
""".split())

def informative_tokens(text: str):
    if not isinstance(text, str):
        return []
    toks = []
    for m in re.findall(r"\b(?:19|20)\d{2}\b", text):
        toks.append(m)
    for m in re.findall(r"\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b", text):
        toks.append(m.lower())
    for m in re.findall(r"\b[A-Z]{2,}\b", text):
        toks.append(m.lower())
    return [t for t in toks if t.lower() not in STOP and len(t) > 2]

REFUSAL_MARK = "don't have information"

def custom_faithfulness(answer: str, contexts):
    if not isinstance(answer, str) or len(answer.strip()) == 0:
        return float("nan")
    # Refusals: no claims to verify -> treat as faithful (1.0)
    if REFUSAL_MARK in answer.lower() or "do not have information" in answer.lower():
        return 1.0
    ans_toks = informative_tokens(answer)
    if len(ans_toks) == 0:
        return float("nan")
    ctx_blob = " ".join(contexts).lower() if contexts else ""
    if len(ctx_blob) == 0:
        return 0.0
    hits = sum(1 for t in ans_toks if t.lower() in ctx_blob)
    return hits / len(ans_toks)

def is_refusal(answer):
    return REFUSAL_MARK in str(answer).lower() or "do not have information" in str(answer).lower()

# Score every row for both prompts
rows = []
for _, r in ablation_c_df.iterrows():
    q = r["query"]
    ctx = contexts_by_query.get(q, [])
    rows.append({
        "category":     r["category"],
        "query":        q,
        "perm_faith":   custom_faithfulness(r["permissive_answer"], ctx),
        "strict_faith": custom_faithfulness(r["strict_answer"], ctx),
        "perm_refusal":   int(is_refusal(r["permissive_answer"])),
        "strict_refusal": int(is_refusal(r["strict_answer"])),
    })

ablation_c_faith_df = pd.DataFrame(rows)
display(ablation_c_faith_df)

# Two summary views:
#  (a) Mean over ALL rows (refusals count as 1.0 — same convention as CELL RAGAS-6)
#  (b) Mean over substantive answers only (refusals excluded, to avoid inflating strict)
def mean_excl_refusals(faith_col, refusal_col):
    mask = ablation_c_faith_df[refusal_col] == 0
    return float(ablation_c_faith_df.loc[mask, faith_col].mean())

ablation_c_faith_summary = pd.DataFrame([
    {
        "prompt_type": "permissive",
        "mean_faithfulness_all":           float(ablation_c_faith_df["perm_faith"].mean()),
        "mean_faithfulness_excl_refusals": mean_excl_refusals("perm_faith", "perm_refusal"),
        "num_refusals":  int(ablation_c_faith_df["perm_refusal"].sum()),
        "num_queries":   len(ablation_c_faith_df),
    },
    {
        "prompt_type": "strict_plus_missing_title_guard",
        "mean_faithfulness_all":           float(ablation_c_faith_df["strict_faith"].mean()),
        "mean_faithfulness_excl_refusals": mean_excl_refusals("strict_faith", "strict_refusal"),
        "num_refusals":  int(ablation_c_faith_df["strict_refusal"].sum()),
        "num_queries":   len(ablation_c_faith_df),
    },
])

print("\nAblation C — Faithfulness per prompt (RAGAS proxy):")
display(ablation_c_faith_summary)

ablation_c_faith_df.to_csv(RESULTS_DIR / "ablation_c_faithfulness_per_prompt.csv", index=False)
ablation_c_faith_summary.to_csv(RESULTS_DIR / "ablation_c_faithfulness_summary.csv", index=False)
print("\nSaved:")
print(" -", RESULTS_DIR / "ablation_c_faithfulness_per_prompt.csv")
print(" -", RESULTS_DIR / "ablation_c_faithfulness_summary.csv")

,category,query,perm_faith,strict_faith,perm_refusal,strict_refusal
0,standard,Who directed Inception?,1.000000,1.000000,0,0
1,standard,Who directed The Matrix?,1.000000,1.000000,0,0
2,standard,What is Avatar about?,0.833333,0.714286,0,0
3,standard,Who directed Titanic?,1.000000,1.000000,0,0
4,standard,What is Finding Nemo about?,1.000000,1.000000,0,0
5,adversarial,A movie about dreams inside dreams and stealin...,1.000000,1.000000,0,0
6,adversarial,A movie about a computer hacker discovering re...,1.000000,1.000000,0,0
7,adversarial,A movie about toys that come alive when humans...,1.000000,1.000000,0,0
8,adversarial,A movie about a clownfish father searching for...,1.000000,1.000000,0,0
9,adversarial,A movie about a ship disaster and a romance ac...,0.500000,1.000000,0,0



Ablation C — Faithfulness per prompt (RAGAS proxy):


,prompt_type,mean_faithfulness_all,mean_faithfulness_excl_refusals,num_refusals,num_queries
0,permissive,0.900000,0.90000,0,20
1,strict_plus_missing_title_guard,0.935714,0.92437,3,20



Saved:
 - /content/drive/MyDrive/movie_rag_results/ablation_c_faithfulness_per_prompt.csv
 - /content/drive/MyDrive/movie_rag_results/ablation_c_faithfulness_summary.csv


In [ ]:
# CELL 30 — Improved explicit-title missing guard

def normalize_title_candidate(text):
    text = normalize_text_for_match(text)
    text = text.strip()
    return text

def candidate_title_exists_in_corpus(candidate):
    """
    Checks whether the extracted candidate title exists as an exact normalized title.
    Also supports simple leading-article variants, e.g. Terminator -> The Terminator.
    """
    if candidate is None:
        return False

    cand = normalize_title_candidate(candidate)

    variants = {
        cand,
        f"the {cand}",
        cand.replace("the ", "", 1) if cand.startswith("the ") else cand
    }

    for v in variants:
        if v in norm_to_records:
            return True

    return False


def is_explicit_title_missing(query):
    """
    If the query appears to ask about a specific title,
    but that extracted title does not exactly exist in the corpus, refuse.
    This prevents partial matches like:
    'Everything Everywhere All at Once' -> 'Everything'
    """
    candidate = extract_candidate_title_from_query(query)

    if candidate is None:
        return False

    return not candidate_title_exists_in_corpus(candidate)


# Quick checks
check_queries = [
    "Who directed Toy Story?",
    "Who directed Oppenheimer?",
    "What is Dune 2021 about?",
    "Who directed Everything Everywhere All at Once?",
    "What is The Completely Fictional Banana Spaceship Movie about?",
    "What is the plot of The Godfather?",
    "What is the plot of Terminator?",
    "What is Avatar about?"
]

for q in check_queries:
    print(
        q,
        "=> candidate:",
        extract_candidate_title_from_query(q),
        "| exists:",
        candidate_title_exists_in_corpus(extract_candidate_title_from_query(q)),
        "| missing:",
        is_explicit_title_missing(q)
    )

Who directed Toy Story? => candidate: toy story | exists: True | missing: False
Who directed Oppenheimer? => candidate: oppenheimer | exists: False | missing: True
What is Dune 2021 about? => candidate: dune 2021 | exists: False | missing: True
Who directed Everything Everywhere All at Once? => candidate: everything everywhere all at once | exists: False | missing: True
What is The Completely Fictional Banana Spaceship Movie about? => candidate: the completely fictional banana spaceship movie | exists: False | missing: True
What is the plot of The Godfather? => candidate: the godfather | exists: True | missing: False
What is the plot of Terminator? => candidate: terminator | exists: True | missing: False
What is Avatar about? => candidate: avatar | exists: True | missing: False


In [ ]:
# CELL 31 — Ablation A: helper for embedding model comparison

def build_faiss_index_for_model(model_name, batch_size=128):
    print("=" * 100)
    print("Loading embedding model:", model_name)

    model = SentenceTransformer(model_name, device=device)

    texts = movies["document_text"].tolist()

    start = time.perf_counter()

    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    encoding_time = time.perf_counter() - start
    docs_per_sec = len(texts) / encoding_time

    dim = embeddings.shape[1]
    memory_gb = embeddings.nbytes / (1024 ** 3)

    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    print("Model:", model_name)
    print("Embedding shape:", embeddings.shape)
    print(f"Encoding time: {encoding_time:.2f} sec")
    print(f"Throughput: {docs_per_sec:.2f} docs/sec")
    print(f"Embedding memory: {memory_gb:.4f} GB")
    print("FAISS vectors:", index.ntotal)

    return model, embeddings, index, {
        "model_name": model_name,
        "embedding_dim": dim,
        "encoding_time_sec": encoding_time,
        "docs_per_sec": docs_per_sec,
        "embedding_memory_gb": memory_gb
    }


def retrieve_with_custom_model(query, model, index, k=20):
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        row = movies.iloc[int(idx)]
        results.append({
            "faiss_id": int(idx),
            "title": row["title"],
            "year": row["release_year"],
            "score": float(score),
            "director": row["director"],
            "cast_text": row["cast_text"],
            "genres_text": row["genres_text"],
            "overview": row["overview"],
            "document_text": row["document_text"]
        })

    return results


def evaluate_embedding_model_retrieval(model_name, model, index, k_retrieve=20, n_final=5):
    rows = []
    latencies = []

    for _, row in test_df.iterrows():
        query = row["query"]
        gold_titles = row["gold_titles"]

        start = time.perf_counter()

        candidates = retrieve_with_custom_model(query, model, index, k=k_retrieve)

        # Same re-ranker held constant
        final_results = rerank_title_aware(query, candidates, n=max(10, n_final))

        latency_ms = (time.perf_counter() - start) * 1000
        latencies.append(latency_ms)

        retrieved_titles = [r["title"] for r in final_results]

        rows.append({
            "model_name": model_name,
            "category": row["category"],
            "query": query,
            "gold_titles": gold_titles,
            "retrieved_titles": retrieved_titles[:10],
            "R@5": hit_at_k(retrieved_titles, gold_titles, 5),
            "MRR": reciprocal_rank(retrieved_titles, gold_titles),
            "latency_ms": latency_ms
        })

    detail_df = pd.DataFrame(rows)
    answerable_df = detail_df[detail_df["gold_titles"].apply(lambda x: len(x) > 0)].copy()

    metrics = {
        "model_name": model_name,
        "R@5": answerable_df["R@5"].mean(),
        "MRR": answerable_df["MRR"].mean(),
        "mean_latency_ms": np.mean(latencies),
        "median_latency_ms": np.median(latencies)
    }

    return detail_df, metrics

In [ ]:
# CELL 32 — Ablation A: small vs base bi-encoder

ablation_a_models = [
    "BAAI/bge-small-en-v1.5",
    "BAAI/bge-base-en-v1.5"
]

ablation_a_summary_rows = []
ablation_a_detail_dfs = []

for model_name in ablation_a_models:
    custom_model, custom_embeddings, custom_index, build_stats = build_faiss_index_for_model(
        model_name=model_name,
        batch_size=128
    )

    detail_df, retrieval_metrics = evaluate_embedding_model_retrieval(
        model_name=model_name,
        model=custom_model,
        index=custom_index,
        k_retrieve=20,
        n_final=5
    )

    summary_row = {}
    summary_row.update(build_stats)
    summary_row.update(retrieval_metrics)

    ablation_a_summary_rows.append(summary_row)
    ablation_a_detail_dfs.append(detail_df)

    # Save model-specific outputs
    safe_name = model_name.replace("/", "_")
    detail_df.to_csv(RESULTS_DIR / f"ablation_a_details_{safe_name}.csv", index=False)

    # Free memory before next model
    del custom_model
    del custom_embeddings
    del custom_index
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

ablation_a_df = pd.DataFrame(ablation_a_summary_rows)
display(ablation_a_df)

ablation_a_df.to_csv(RESULTS_DIR / "ablation_a_biencoder_size.csv", index=False)

Loading embedding model: BAAI/bge-small-en-v1.5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/324 [00:00<?, ?it/s]

Model: BAAI/bge-small-en-v1.5
Embedding shape: (41367, 384)
Encoding time: 147.08 sec
Throughput: 281.25 docs/sec
Embedding memory: 0.0592 GB
FAISS vectors: 41367
Loading embedding model: BAAI/bge-base-en-v1.5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/324 [00:00<?, ?it/s]

Model: BAAI/bge-base-en-v1.5
Embedding shape: (41367, 768)
Encoding time: 422.38 sec
Throughput: 97.94 docs/sec
Embedding memory: 0.1184 GB
FAISS vectors: 41367


,model_name,embedding_dim,encoding_time_sec,docs_per_sec,embedding_memory_gb,R@5,MRR,mean_latency_ms,median_latency_ms
0,BAAI/bge-small-en-v1.5,384,147.083009,281.249346,0.059176,0.75,0.71875,1501.257831,1421.32587
1,BAAI/bge-base-en-v1.5,768,422.380491,97.937762,0.118352,0.75,0.71875,1508.119823,1437.93957


In [ ]:
# CELL 33 — Failure Diary / Bad Cases Table

failure_cases = [
    {
        "Input / Query": "What is Avatar about?",
        "Model Output": "Before fix, the system retrieved My Avatar and Me above Avatar and generated an answer with unsupported details.",
        "Gold / Correct Answer": "Avatar is about Jake Sully, a paraplegic Marine who enters the world of Pandora and becomes involved with the Na'vi.",
        "Error Type": "Retrieval Miss, Re-ranker Failure, Hallucination",
        "Root Cause Analysis": "The query title 'Avatar' was present in multiple movie titles. The re-ranker placed 'My Avatar and Me' above the exact title match, which caused the generator to mix correct and unsupported details.",
        "Implemented Fix": "Added title-aware re-ranking boost for exact title matches.",
        "Before Fix Evidence": "Top-1 retrieved title was My Avatar and Me.",
        "After Fix Evidence": "Top-1 retrieved title became Avatar; final answer was grounded in Avatar."
    },
    {
        "Input / Query": "Who directed Alien?",
        "Model Output": "Before fix, the normal re-ranker ranked Alien 2: On Earth above Alien.",
        "Gold / Correct Answer": "Ridley Scott directed Alien.",
        "Error Type": "Sequel Confusion, Re-ranker Failure",
        "Root Cause Analysis": "Many movies share the token Alien. The bi-encoder and re-ranker confused the original film with related titles and sequels.",
        "Implemented Fix": "Added safer title-aware matching and canonical title handling to avoid matching Alien³ instead of Alien.",
        "Before Fix Evidence": "Normal re-ranker top-1 was Alien 2: On Earth.",
        "After Fix Evidence": "Title-aware re-ranker top-1 became Alien; final answer was Ridley Scott directed Alien."
    },
    {
        "Input / Query": "Who directed Oppenheimer?",
        "Model Output": "Before strict guard, the system answered using The Day After Trinity and said Jon Else.",
        "Gold / Correct Answer": "The corpus has no Oppenheimer movie entry because the dataset ends before July 2017; the system should refuse.",
        "Error Type": "Hallucination, Unanswerable Query, Retrieval Miss",
        "Root Cause Analysis": "The retriever found semantically related Oppenheimer documentary/context documents, but not the requested post-2017 movie. The permissive prompt answered from the wrong context.",
        "Implemented Fix": "Added explicit-title missing guard and strict refusal prompt.",
        "Before Fix Evidence": "Permissive answer: Jon Else.",
        "After Fix Evidence": "Strict answer: I don't have information about that in the provided context."
    },
    {
        "Input / Query": "A movie about dreams inside dreams and stealing secrets from the mind",
        "Model Output": "The system retrieved Dreamscape instead of Inception.",
        "Gold / Correct Answer": "Inception",
        "Error Type": "Semantic Drift, Retrieval Miss",
        "Root Cause Analysis": "The query was descriptive and did not contain the exact title. The retriever matched general dream-related vocabulary rather than the intended movie.",
        "Implemented Fix": "No final fix implemented; this remains a limitation of the current retrieval setup.",
        "Before Fix Evidence": "Retrieved titles did not include Inception in the top-5.",
        "After Fix Evidence": "Not fixed; documented as an adversarial semantic failure."
    }
]

failure_diary_df = pd.DataFrame(failure_cases)
display(failure_diary_df)

failure_diary_df.to_csv(RESULTS_DIR / "failure_diary.csv", index=False)

,Input / Query,Model Output,Gold / Correct Answer,Error Type,Root Cause Analysis,Implemented Fix,Before Fix Evidence,After Fix Evidence
0,What is Avatar about?,"Before fix, the system retrieved My Avatar and...","Avatar is about Jake Sully, a paraplegic Marin...","Retrieval Miss, Re-ranker Failure, Hallucination",The query title 'Avatar' was present in multip...,Added title-aware re-ranking boost for exact t...,Top-1 retrieved title was My Avatar and Me.,Top-1 retrieved title became Avatar; final ans...
1,Who directed Alien?,"Before fix, the normal re-ranker ranked Alien ...",Ridley Scott directed Alien.,"Sequel Confusion, Re-ranker Failure",Many movies share the token Alien. The bi-enco...,Added safer title-aware matching and canonical...,Normal re-ranker top-1 was Alien 2: On Earth.,Title-aware re-ranker top-1 became Alien; fina...
2,Who directed Oppenheimer?,"Before strict guard, the system answered using...",The corpus has no Oppenheimer movie entry beca...,"Hallucination, Unanswerable Query, Retrieval Miss",The retriever found semantically related Oppen...,Added explicit-title missing guard and strict ...,Permissive answer: Jon Else.,Strict answer: I don't have information about ...
3,A movie about dreams inside dreams and stealin...,The system retrieved Dreamscape instead of Inc...,Inception,"Semantic Drift, Retrieval Miss",The query was descriptive and did not contain ...,No final fix implemented; this remains a limit...,Retrieved titles did not include Inception in ...,Not fixed; documented as an adversarial semant...


In [ ]:
# CELL 34 — Prepare RAGAS evaluation data

# save context for ragas
ragas_rows = []

for _, row in test_df.iterrows():
    query = row["query"]

    candidates = retrieve(query, k=20)
    reranked_results = rerank_title_aware(query, candidates, n=5)

    result = run_query(query)

    contexts = [
        r["document_text"]
        for r in reranked_results
    ]


    if len(row["gold_titles"]) > 0:
        reference = f"The answer should be grounded in: {', '.join(row['gold_titles'])}."
    else:
        reference = "I don't have information about that in the provided context."

    ragas_rows.append({
        "question": query,
        "answer": result["answer"],
        "contexts": contexts,
        "reference": reference,
        "category": row["category"],
        "gold_titles": row["gold_titles"],
        "retrieved_titles": result["retrieved_titles"]
    })

ragas_eval_df = pd.DataFrame(ragas_rows)
display(ragas_eval_df[["category", "question", "answer", "reference", "retrieved_titles"]])

ragas_eval_df.to_pickle(RESULTS_DIR / "ragas_eval_df.pkl")
ragas_eval_df.to_csv(RESULTS_DIR / "ragas_eval_df_preview.csv", index=False)

print("Saved RAGAS eval data.")

[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

,category,question,answer,reference,retrieved_titles
0,standard,Who directed Inception?,Christopher Nolan directed Inception.,The answer should be grounded in: Inception.,"[Inception, Transcendent Man, Directed by Sidn..."
1,standard,Who directed The Matrix?,The Matrix was directed by Lana Wachowski.,The answer should be grounded in: The Matrix.,"[The Matrix, The Matrix Revisited, Return to S..."
2,standard,What is Avatar about?,Avatar is an action-adventure science fiction ...,The answer should be grounded in: Avatar.,"[Avatar, My Avatar and Me, Avatar: Creating th..."
3,standard,Who directed Titanic?,James Cameron directed Titanic.,The answer should be grounded in: Titanic.,"[Titanic, Titanic 2, Raise the Titanic, Titani..."
4,standard,What is Finding Nemo about?,Finding Nemo is about a young clownfish named ...,The answer should be grounded in: Finding Nemo.,"[Finding Nemo, Finding Dory, Captain Nemo and ..."
5,adversarial,A movie about dreams inside dreams and stealin...,I don't have information about that in the pro...,The answer should be grounded in: Inception.,"[Dreamscape, Passion of Mind, Meshes of the Af..."
6,adversarial,A movie about a computer hacker discovering re...,I don't have information about that in the pro...,The answer should be grounded in: The Matrix.,"[The Zero Theorem, War Games: The Dead Code, A..."
7,adversarial,A movie about toys that come alive when humans...,I don't have information about that in the pro...,The answer should be grounded in: Toy Story.,"[Silent Night, Deadly Night 5: The Toy Maker, ..."
8,adversarial,A movie about a clownfish father searching for...,Finding Nemo,The answer should be grounded in: Finding Nemo.,"[Finding Nemo, Fathers' Day, Our Father, Fathe..."
9,adversarial,A movie about a ship disaster and a romance ac...,I don't have information about that in the pro...,The answer should be grounded in: Titanic.,"[Sea Wife, Message in a Bottle, Stormy Waters,..."


Saved RAGAS eval data.


In [ ]:
# CELL 35 — Prepare small RAGAS subset first

ragas_small_df = ragas_eval_df.sample(
    n=8,
    random_state=SEED
).reset_index(drop=True)

display(ragas_small_df[["category", "question", "answer", "reference", "retrieved_titles"]])

,category,question,answer,reference,retrieved_titles
0,standard,Who directed Inception?,Christopher Nolan directed Inception.,The answer should be grounded in: Inception.,"[Inception, Transcendent Man, Directed by Sidn..."
1,sequel_confusion,What is the plot of Terminator?,The plot of Terminator revolves around John Co...,The answer should be grounded in: The Terminator.,"[Terminator Genisys, The Terminator, The Termi..."
2,sequel_confusion,What is the plot of The Godfather?,The plot of The Godfather revolves around the ...,The answer should be grounded in: The Godfather.,"[The Godfather, The Godfather: Part III, The G..."
3,standard,Who directed The Matrix?,The Matrix was directed by Lana Wachowski.,The answer should be grounded in: The Matrix.,"[The Matrix, The Matrix Revisited, Return to S..."
4,adversarial,A movie about a clownfish father searching for...,Finding Nemo,The answer should be grounded in: Finding Nemo.,"[Finding Nemo, Fathers' Day, Our Father, Fathe..."
5,adversarial,A movie about dreams inside dreams and stealin...,I don't have information about that in the pro...,The answer should be grounded in: Inception.,"[Dreamscape, Passion of Mind, Meshes of the Af..."
6,unanswerable,What is Dune 2021 about?,I don't have information about that in the pro...,I don't have information about that in the pro...,"[Dune, Frank Herbert's Dune, Jodorowsky's Dune..."
7,standard,Who directed Titanic?,James Cameron directed Titanic.,The answer should be grounded in: Titanic.,"[Titanic, Titanic 2, Raise the Titanic, Titani..."


In [ ]:
# CELL 36 — RAGAS imports

from datasets import Dataset

try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision
    print("RAGAS imported successfully.")
except Exception as e:
    print("RAGAS import error:")
    print(type(e).__name__, e)

RAGAS imported successfully.


In [ ]:
# CELL 37 — Convert dataframe to RAGAS Dataset

ragas_dataset = Dataset.from_dict({
    "question": ragas_small_df["question"].tolist(),
    "answer": ragas_small_df["answer"].tolist(),
    "contexts": ragas_small_df["contexts"].tolist(),
    "ground_truth": ragas_small_df["reference"].tolist(),
    "reference": ragas_small_df["reference"].tolist()
})

print(ragas_dataset)
print(ragas_dataset[0])

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth', 'reference'],
    num_rows: 8
})
{'question': 'Who directed Inception?', 'answer': 'Christopher Nolan directed Inception.', 'contexts': ['Title: Inception\nYear: 2010\nGenres: Action, Thriller, Science Fiction, Mystery, Adventure\nDirector: Christopher Nolan\nCast: Leonardo DiCaprio, Joseph Gordon-Levitt, Ellen Page, Tom Hardy, Ken Watanabe\nOverview: Cobb, a skilled thief who commits corporate espionage by infiltrating the subconscious of his targets is offered a chance to regain his old life as payment for a task considered to be impossible: "inception", the implantation of another person\'s idea into a target\'s subconscious.', 'Title: Transcendent Man\nYear: 2009\nGenres: Documentary\nDirector: Robert Barry Ptolemy\nCast: Tom Abate, Hugo De Garis, Peter Diamandis, Ray Kurzweil\nOverview: The compelling feature-length documentary film, by director Barry Ptolemy, chronicles the life and controversial ideas of 

In [ ]:
# CELL 38 — Try RAGAS evaluation

try:
    ragas_result = evaluate(
        ragas_dataset,
        metrics=[
            faithfulness,
            answer_relevancy,
            context_precision
        ]
    )

    print(ragas_result)
    ragas_result_df = ragas_result.to_pandas()
    display(ragas_result_df)

    ragas_result_df.to_csv(RESULTS_DIR / "ragas_small_results.csv", index=False)

except Exception as e:
    print("RAGAS evaluation failed.")
    print("Error type:", type(e).__name__)
    print("Error message:", e)

RAGAS evaluation failed.
Error type: OpenAIError
Error message: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable


In [ ]:
# CELL 38B — Install wrappers for local RAGAS judge

!pip -q install -U langchain langchain-huggingface

In [ ]:
# CELL 39 — Local RAGAS wrappers

from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings

try:
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper

    local_lc_llm = HuggingFacePipeline(pipeline=generator)

    local_ragas_llm = LangchainLLMWrapper(local_lc_llm)

    local_lc_embeddings = HuggingFaceEmbeddings(
        model_name=EMBED_MODEL_NAME,
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": True}
    )

    local_ragas_embeddings = LangchainEmbeddingsWrapper(local_lc_embeddings)

    print("Local RAGAS LLM and embeddings are ready.")

except Exception as e:
    print("Wrapper setup failed.")
    print("Error type:", type(e).__name__)
    print("Error message:", e)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Local RAGAS LLM and embeddings are ready.


In [ ]:
# CELL 40 — RAGAS evaluation with local judge

try:
    ragas_result = evaluate(
        ragas_dataset,
        metrics=[
            faithfulness,
            answer_relevancy,
            context_precision
        ],
        llm=local_ragas_llm,
        embeddings=local_ragas_embeddings
    )

    print(ragas_result)

    ragas_result_df = ragas_result.to_pandas()
    display(ragas_result_df)

    ragas_result_df.to_csv(RESULTS_DIR / "ragas_small_results_local.csv", index=False)

except Exception as e:
    print("Local RAGAS evaluation failed.")
    print("Error type:", type(e).__name__)
    print("Error message:", e)

Evaluating:   0%|          | 0/24 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

{'faithfulness': nan, 'answer_relevancy': nan, 'context_precision': nan}


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision
0,Who directed Inception?,"[Title: Inception\nYear: 2010\nGenres: Action,...",Christopher Nolan directed Inception.,The answer should be grounded in: Inception.,NaN,NaN,NaN
1,What is the plot of Terminator?,[Title: Terminator Genisys\nYear: 2015\nGenres...,The plot of Terminator revolves around John Co...,The answer should be grounded in: The Terminator.,NaN,NaN,NaN
2,What is the plot of The Godfather?,[Title: The Godfather\nYear: 1972\nGenres: Dra...,The plot of The Godfather revolves around the ...,The answer should be grounded in: The Godfather.,NaN,NaN,NaN
3,Who directed The Matrix?,[Title: The Matrix\nYear: 1999\nGenres: Action...,The Matrix was directed by Lana Wachowski.,The answer should be grounded in: The Matrix.,NaN,NaN,NaN
4,A movie about a clownfish father searching for...,[Title: Finding Nemo\nYear: 2003\nGenres: Anim...,Finding Nemo,The answer should be grounded in: Finding Nemo.,NaN,NaN,NaN
5,A movie about dreams inside dreams and stealin...,[Title: Dreamscape\nYear: 1984\nGenres: Advent...,I don't have information about that in the pro...,The answer should be grounded in: Inception.,NaN,NaN,NaN
6,What is Dune 2021 about?,"[Title: Dune\nYear: 1984\nGenres: Action, Scie...",I don't have information about that in the pro...,I don't have information about that in the pro...,NaN,NaN,NaN
7,Who directed Titanic?,"[Title: Titanic\nYear: 1997\nGenres: Drama, Ro...",James Cameron directed Titanic.,The answer should be grounded in: Titanic.,NaN,NaN,NaN


In [ ]:
# FINAL CHECK — Saved outputs and run_query sanity

print("Movies:", movies.shape)
print("FAISS vectors:", faiss_index.ntotal)

print("\nrun_query sanity check:")
for q in [
    "Who directed Inception?",
    "Who directed Oppenheimer?",
    "Who directed Alien?",
    "What is Avatar about?"
]:
    print("=" * 80)
    print(q)
    print(run_query(q))

print("\nSaved result files:")
for p in sorted(RESULTS_DIR.glob("*")):
    print(p.name)

Movies: (41367, 33)
FAISS vectors: 41367

run_query sanity check:
Who directed Inception?


[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'answer': 'Christopher Nolan directed Inception.', 'retrieved_titles': ['Inception', 'Transcendent Man', 'Directed by Sidney Lumet: How the Devil Was Made', 'Dream Work', 'Hollywood between Paranoia and Sci-Fi. The Power of Myth']}
Who directed Oppenheimer?
{'answer': "I don't have information about that in the provided context.", 'retrieved_titles': ['The Day After Trinity', 'Alchemy', 'Day One', 'Tar', 'Directed by Sidney Lumet: How the Devil Was Made']}
Who directed Alien?


[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'answer': 'Ridley Scott directed Alien.', 'retrieved_titles': ['Alien', 'Alien 2: On Earth', 'The Alien Saga', 'Alien Origin', 'Alien Apocalypse']}
What is Avatar about?


[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'answer': 'Avatar is an action-adventure science fiction film directed by James Cameron. It tells the story of Jake Sully, a paraplegic Marine sent to the moon Pandora on a mission to protect an indigenous alien civilization from exploitation by humans.', 'retrieved_titles': ['Avatar', 'My Avatar and Me', 'Avatar: Creating the World of Pandora', 'Avatar 2', 'The Last Airbender']}

Saved result files:
ablation_a_biencoder_size.csv
ablation_a_details_BAAI_bge-base-en-v1.5.csv
ablation_a_details_BAAI_bge-small-en-v1.5.csv
ablation_b_topk_sweep.csv
ablation_c_prompt_outputs.csv
ablation_c_summary.csv
end_to_end_outputs.csv
failure_diary.csv
ragas_eval_df.pkl
ragas_eval_df_preview.csv
ragas_small_results_local.csv
retrieval_metrics_summary.csv
title_aware_retrieval_details.csv


## RAGAS Evaluation — Fixed Pipeline (Qwen2.5-7B Judge, 4-bit)

The earlier RAGAS cells (CELL 38, 38B, 39, 40) failed for two reasons:

1. The default RAGAS LLM is OpenAI, which is **out of scope** for this project (local models only, no hosted APIs).
2. Wrapping the small Qwen2.5-1.5B generator as the judge produced `TimeoutError` on every job, because a 1.5B model cannot reliably emit the structured JSON that RAGAS prompts require, and the underlying `HuggingFacePipeline` had `max_length=20` defaults that cut off RAGAS's prompt before it could finish.

The fix below follows the standard approach for this situation: save answers before evaluating, restart the runtime, then load only the judge LLM without the generator LLM:

- I already saved `ragas_eval_df.pkl` to `RESULTS_DIR` in CELL 34.
- **Action required: restart the Colab runtime now** (Runtime → Restart session) to free the generator, reranker, and embedding model from GPU memory.
- Run the cells below in order. They load only the judge (Qwen2.5-7B-Instruct in 4-bit, ~4.5 GB on T4) and a small embedding model for `answer_relevancy`.
- Timeouts are extended and `max_workers=1` so the judge does not get overwhelmed.
- A custom keyword-based faithfulness fallback is computed in parallel, so the report has at least one reliable grounding signal even if RAGAS itself struggles.

In [ ]:
# CELL RAGAS-1 — Setup after runtime restart (judge-only environment)

# Run this AFTER restarting the Colab runtime.
# This cell re-imports the minimum needed and reloads the saved evaluation data.

from google.colab import drive
drive.mount('/content/drive')

import os
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

SEED = 42

RESULTS_DIR = Path("/content/drive/MyDrive/movie_rag_results")
INDEX_DIR = Path("/content/drive/MyDrive/movie_rag_faiss")

# Load the evaluation dataframe we built before restart
ragas_eval_df = pd.read_pickle(RESULTS_DIR / "ragas_eval_df.pkl")

print("Loaded ragas_eval_df:", ragas_eval_df.shape)
print("Columns:", list(ragas_eval_df.columns))
print("\nCategories:")
print(ragas_eval_df["category"].value_counts())

# GPU sanity
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    free_gb = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
    print(f"\nFree GPU memory after restart: {free_gb:.2f} GB")
else:
    print("\nNo GPU detected.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded ragas_eval_df: (20, 7)
Columns: ['question', 'answer', 'contexts', 'reference', 'category', 'gold_titles', 'retrieved_titles']

Categories:
category
standard            7
adversarial         5
unanswerable        4
sequel_confusion    4
Name: count, dtype: int64

Free GPU memory after restart: 15.64 GB


In [ ]:
# CELL RAGAS-2 — Install required libraries for RAGAS + 4-bit judge

!pip -q install -U "ragas==0.2.10" "datasets" "langchain" "langchain-huggingface" "langchain-community"
!pip -q install -U "bitsandbytes" "accelerate" "transformers"

print("Libraries installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.7/175.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 4.7 MB/s eta 0:00:00
Libraries installed.


In [ ]:
# CELL RAGAS-3 — Load Qwen2.5-7B-Instruct judge in 4-bit (NF4)

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

JUDGE_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading judge tokenizer...")
judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_NAME)
if judge_tokenizer.pad_token_id is None:
    judge_tokenizer.pad_token_id = judge_tokenizer.eos_token_id

print("Loading judge model in 4-bit (this takes ~3-4 minutes)...")
judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

# Pipeline tuned for RAGAS: long enough output, deterministic, batched off
judge_pipe = pipeline(
    "text-generation",
    model=judge_model,
    tokenizer=judge_tokenizer,
    max_new_tokens=512,           # CRITICAL: RAGAS needs long structured outputs
    do_sample=False,
    return_full_text=False,
    pad_token_id=judge_tokenizer.eos_token_id,
)

print("Judge ready:", JUDGE_MODEL_NAME)
print("GPU memory used:", torch.cuda.memory_allocated() / 1e9, "GB")

Loading judge tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Judge ready: Qwen/Qwen2.5-7B-Instruct
GPU memory used: 5.559835648 GB


In [ ]:
# CELL RAGAS-4 — Wrap judge for RAGAS, load small embeddings for answer_relevancy

from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# Wrap the HF pipeline so LangChain treats it as an LLM
judge_lc = HuggingFacePipeline(pipeline=judge_pipe)
judge_ragas = LangchainLLMWrapper(judge_lc)

# RAGAS answer_relevancy needs an embedding model. Use the small one (fast, low VRAM).
embed_lc = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)
embed_ragas = LangchainEmbeddingsWrapper(embed_lc)

print("Judge + embeddings wrapped for RAGAS.")
print("GPU memory used:", torch.cuda.memory_allocated() / 1e9, "GB")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Judge + embeddings wrapped for RAGAS.
GPU memory used: 5.69328384 GB


In [ ]:
# CELL RAGAS-5 — Run RAGAS on the small subset with safe RunConfig

from datasets import Dataset
from ragas import evaluate
from ragas.run_config import RunConfig
from ragas.metrics import faithfulness, answer_relevancy, context_precision

# Sample 8 queries (balanced if possible)
ragas_small_df = ragas_eval_df.sample(n=8, random_state=SEED).reset_index(drop=True)
print("Subset categories:")
print(ragas_small_df["category"].value_counts())

# RAGAS expects exactly these column names
ragas_dataset = Dataset.from_dict({
    "question":    ragas_small_df["question"].tolist(),
    "answer":      ragas_small_df["answer"].tolist(),
    "contexts":    ragas_small_df["contexts"].tolist(),
    "ground_truth": ragas_small_df["reference"].tolist(),
    "reference":   ragas_small_df["reference"].tolist(),
})

# CRITICAL: serialize calls (max_workers=1) and extend timeout, otherwise the
# 7B judge will be hammered by parallel requests and fall over.
run_cfg = RunConfig(
    timeout=300,       # 5 min per call — generous for 4-bit on T4
    max_retries=3,
    max_workers=1,     # serial: one judge call at a time
)

print("\nRunning RAGAS evaluation (expect ~10-15 minutes)...")
ragas_result = evaluate(
    ragas_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision],
    llm=judge_ragas,
    embeddings=embed_ragas,
    run_config=run_cfg,
    raise_exceptions=False,   # don't crash on a single bad row
)

print("\nRAGAS aggregate scores:")
print(ragas_result)

ragas_result_df = ragas_result.to_pandas()
ragas_result_df["category"] = ragas_small_df["category"].values
display(ragas_result_df)

ragas_result_df.to_csv(RESULTS_DIR / "ragas_results_qwen7b_judge.csv", index=False)
print("\nSaved to:", RESULTS_DIR / "ragas_results_qwen7b_judge.csv")

Subset categories:
category
standard            3
sequel_confusion    2
adversarial         2
unanswerable        1
Name: count, dtype: int64

Running RAGAS evaluation (expect ~10-15 minutes)...


Evaluating:   0%|          | 0/24 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the docum


RAGAS aggregate scores:
{'faithfulness': 0.7143, 'answer_relevancy': 0.7128, 'context_precision': 0.1875}


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,category
0,Who directed Inception?,"[Title: Inception\nYear: 2010\nGenres: Action,...",Christopher Nolan directed Inception.,The answer should be grounded in: Inception.,1.0,0.978040,0.0,standard
1,What is the plot of Terminator?,[Title: Terminator Genisys\nYear: 2015\nGenres...,The plot of Terminator revolves around John Co...,The answer should be grounded in: The Terminator.,0.0,0.856484,0.5,sequel_confusion
2,What is the plot of The Godfather?,[Title: The Godfather\nYear: 1972\nGenres: Dra...,The plot of The Godfather revolves around the ...,The answer should be grounded in: The Godfather.,NaN,0.949649,0.0,sequel_confusion
3,Who directed The Matrix?,[Title: The Matrix\nYear: 1999\nGenres: Action...,The Matrix was directed by Lana Wachowski.,The answer should be grounded in: The Matrix.,1.0,1.000000,0.0,standard
4,A movie about a clownfish father searching for...,[Title: Finding Nemo\nYear: 2003\nGenres: Anim...,Finding Nemo,The answer should be grounded in: Finding Nemo.,1.0,0.942061,1.0,adversarial
5,A movie about dreams inside dreams and stealin...,[Title: Dreamscape\nYear: 1984\nGenres: Advent...,I don't have information about that in the pro...,The answer should be grounded in: Inception.,0.0,0.000000,0.0,adversarial
6,What is Dune 2021 about?,"[Title: Dune\nYear: 1984\nGenres: Action, Scie...",I don't have information about that in the pro...,I don't have information about that in the pro...,1.0,0.000000,0.0,unanswerable
7,Who directed Titanic?,"[Title: Titanic\nYear: 1997\nGenres: Drama, Ro...",James Cameron directed Titanic.,The answer should be grounded in: Titanic.,1.0,0.976031,0.0,standard



Saved to: /content/drive/MyDrive/movie_rag_results/ragas_results_qwen7b_judge.csv


In [ ]:
# CELL RAGAS-6 — Custom faithfulness fallback (works even if RAGAS partially fails)

# Idea: a claim in the answer is "faithful" if its informative tokens (proper nouns,
# years, capitalised multi-word phrases) all appear somewhere in the retrieved contexts.
# This is a coarse signal, but it cannot return NaN and gives the report a baseline number.

import re
from collections import Counter

STOP = set("""
a an the and or but if then so of to in on at for from with by as is are was were be been being
this that these those it its his her their there here what which who whom whose how why when
about against between into through during before after above below up down out off over under
i you he she we they me him us them my your its our ought would could should may might can
""".split())

def informative_tokens(text: str):
    if not isinstance(text, str):
        return []
    # capture: 4-digit years, capitalised words/phrases, all-caps acronyms
    toks = []
    for m in re.findall(r"\b(?:19|20)\d{2}\b", text):
        toks.append(m)
    for m in re.findall(r"\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b", text):
        toks.append(m.lower())
    for m in re.findall(r"\b[A-Z]{2,}\b", text):
        toks.append(m.lower())
    # filter stopwords and very short
    return [t for t in toks if t.lower() not in STOP and len(t) > 2]

def custom_faithfulness(answer: str, contexts):
    """Fraction of informative answer tokens that appear in any retrieved context."""
    if not isinstance(answer, str) or len(answer.strip()) == 0:
        return float("nan")
    # Refusals get a free pass (no claims to verify)
    if "don't have information" in answer.lower() or "do not have information" in answer.lower():
        return 1.0
    ans_toks = informative_tokens(answer)
    if len(ans_toks) == 0:
        return float("nan")
    ctx_blob = " ".join(contexts).lower() if contexts else ""
    if len(ctx_blob) == 0:
        return 0.0
    hits = sum(1 for t in ans_toks if t.lower() in ctx_blob)
    return hits / len(ans_toks)

# Compute on FULL eval df (all 20 queries, not just the 8 in RAGAS subset)
ragas_eval_df["custom_faithfulness"] = ragas_eval_df.apply(
    lambda r: custom_faithfulness(r["answer"], r["contexts"]),
    axis=1,
)

faith_by_cat = ragas_eval_df.groupby("category")["custom_faithfulness"].agg(["mean", "count"])
print("Custom faithfulness by category (full 20-query test set):")
print(faith_by_cat)

print("\nOverall mean custom faithfulness:", ragas_eval_df["custom_faithfulness"].mean())

ragas_eval_df.to_csv(RESULTS_DIR / "ragas_eval_df_with_custom_faithfulness.csv", index=False)

Custom faithfulness by category (full 20-query test set):
                      mean  count
category                         
adversarial       1.000000      5
sequel_confusion  1.000000      4
standard          0.971429      7
unanswerable      1.000000      4

Overall mean custom faithfulness: 0.99


In [ ]:
# CELL RAGAS-7 — Final summary combining RAGAS + custom faithfulness

# Aggregate RAGAS results (handle NaN gracefully)
def safe_mean(s):
    s = pd.to_numeric(s, errors="coerce")
    return float(s.mean()) if s.notna().any() else float("nan")

ragas_summary = {
    "ragas_faithfulness_mean":    safe_mean(ragas_result_df.get("faithfulness", pd.Series(dtype=float))),
    "ragas_answer_relevancy_mean": safe_mean(ragas_result_df.get("answer_relevancy", pd.Series(dtype=float))),
    "ragas_context_precision_mean": safe_mean(ragas_result_df.get("context_precision", pd.Series(dtype=float))),
    "custom_faithfulness_mean_full20": float(ragas_eval_df["custom_faithfulness"].mean()),
    "ragas_subset_size": int(len(ragas_result_df)),
    "full_test_set_size": int(len(ragas_eval_df)),
    "judge_model": "Qwen/Qwen2.5-7B-Instruct (4-bit NF4)",
}

print("=" * 70)
print("FINAL RAGAS EVALUATION SUMMARY")
print("=" * 70)
for k, v in ragas_summary.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

# Save
with open(RESULTS_DIR / "ragas_summary.json", "w") as f:
    json.dump(ragas_summary, f, indent=2)

print("\nSaved final summary to:", RESULTS_DIR / "ragas_summary.json")

FINAL RAGAS EVALUATION SUMMARY
  ragas_faithfulness_mean: 0.7143
  ragas_answer_relevancy_mean: 0.7128
  ragas_context_precision_mean: 0.1875
  custom_faithfulness_mean_full20: 0.9900
  ragas_subset_size: 8
  full_test_set_size: 20
  judge_model: Qwen/Qwen2.5-7B-Instruct (4-bit NF4)

Saved final summary to: /content/drive/MyDrive/movie_rag_results/ragas_summary.json


# Analysis Report

This report covers data pipeline statistics, model and VectorDB justifications, ablation interpretations, evaluation discussion, and lessons learned. All numbers are taken directly from the cells above.

---

## 1. Data Pipeline Summary

After the preprocessing steps in CELL 4 (drop NaN titles, drop empty overviews, deduplicate titles by keeping the longer overview, parse genres/director/cast):

- **Initial movies**: 45,466
- **After valid numeric id**: 45,463
- **After dropping missing titles**: −3
- **After dropping empty overviews**: −959
- **After merging with credits**: 44,577
- **After deduplicating titles**: −3,210
- **Final corpus size**: **41,367 movies**

**Overview length** (whitespace-token estimate): mean 56.4, median 50, max 187 tokens.
**document_text length**: mean 79.9, median 73, max 212 tokens.

**document_text format**: I concatenate `Title`, `Year`, `Genres`, `Director`, `Cast` (top 5), and `Overview` into a single labelled block per movie. Embedding the title and year directly inside `document_text` (rather than only in metadata) is the design choice that makes the bi-encoder and cross-encoder competitive on title-based queries — without it, sequel-confusion queries like "Who directed Toy Story?" silently retrieve "Toy Story 3" because the original's overview is shorter.

I used the **full preprocessed corpus** (no random subsampling) for indexing, retrieval, and all evaluation, to avoid a franchise sample-bias trap.

---

## 2. Model Choices and Justifications

### 2.1 Bi-encoder: `BAAI/bge-small-en-v1.5`
Chosen because (a) it is a top-of-its-class small English embedding model on the MTEB retrieval leaderboard, (b) at 384 dims and ~33M params it encodes the full 41K corpus in ~2.5 minutes on T4 (~281 docs/sec), and (c) Ablation A confirmed empirically that the larger `bge-base` checkpoint gives no retrieval gain at this corpus size, so paying 2× memory and 3× indexing time is not justified.

### 2.2 Cross-encoder: `cross-encoder/ms-marco-MiniLM-L-6-v2`
Standard, well-tuned MS MARCO re-ranker with very low VRAM cost (~80MB). It gives a clear precision boost on top-k lists without becoming a bottleneck (re-ranking 20 candidates adds only ~150ms per query on T4).

### 2.3 Generator LLM: `Qwen/Qwen2.5-1.5B-Instruct`, fp16
Chosen because it is the smallest model in the recommended Qwen2.5 family that still reliably follows instruction templates and refusal rules. fp16 (no quantization) was viable because the model is already small (~3GB on GPU), leaving room for the embedding model and re-ranker simultaneously. I evaluated 3B briefly but the GPU-memory-vs-quality tradeoff was not worth it given that Ablation C showed the **prompt** (strict vs permissive) drives quality far more than model size at this scale.

**Hosted APIs were not used at any step of the pipeline** — this was a hard design constraint from the start, everything runs on local/open-source models. The only place a larger model is loaded is the RAGAS judge (Section 6), and that is an evaluation tool external to the pipeline, run after a runtime restart.

### 2.4 RAGAS judge: `Qwen/Qwen2.5-7B-Instruct`, 4-bit NF4
For evaluation only. The 1.5B generator failed as a judge (every job timed out, all metrics returned NaN — see Section 6). 7B in 4-bit fits in ~5.5GB on T4 and reliably emits the structured JSON RAGAS prompts require.

### 2.5 VectorDB: FAISS (`IndexFlatIP` over normalized vectors)
Chosen over ChromaDB for speed and explicit control. `IndexFlatIP` with L2-normalized vectors is mathematically equivalent to cosine similarity and is exact (no approximation error) — important when measuring re-ranker gains, because any ANN approximation noise would confound the ablation. The cost is maintaining an ID-to-metadata mapping in pandas myself, which is ~20 lines of glue code in CELL 8 and a price worth paying for reproducibility. At 41K vectors × 384 dims, query latency is well under 10ms regardless of database choice.

---

## 3. Ablation Results

### 3.1 Ablation A — Bi-encoder size

| Model | Dim | Encoding time (s) | Throughput (docs/s) | Memory (GB) | R@5 | MRR | Median latency (ms) |
|-------|----:|------------------:|--------------------:|------------:|----:|----:|--------------------:|
| bge-small-en-v1.5 | 384 | 147.1 | 281.3 | 0.059 | 0.7500 | 0.7188 | 1421 |
| bge-base-en-v1.5  | 768 | 422.4 |  97.9 | 0.118 | 0.7500 | 0.7188 | 1438 |

**Interpretation.** The base model gives **identical retrieval quality** (R@5=0.75, MRR=0.7188) at almost 3× the encoding time and 2× the embedding memory. At this corpus size (~41K) and with the cross-encoder doing the heavy lifting on the final ranking, the bi-encoder is already saturating the re-ranker's input pool — adding embedding capacity does not help. This is a clean empirical case for "use the smaller model unless you have a measured reason not to."

### 3.2 Ablation B — Top-k sweep before re-ranker

| top-k | R@5 | MRR | Mean latency (ms) | Median latency (ms) |
|------:|----:|----:|------------------:|--------------------:|
|  5 | 0.6875 | 0.6563 | 1465 | 1368 |
| 10 | 0.7500 | 0.7188 | 1485 | 1382 |
| 20 | 0.7500 | 0.7188 | 1661 | 1422 |

**Interpretation.** R@5 jumps from k=5 → k=10 (the re-ranker needs headroom; with only 5 candidates it cannot rescue queries where the gold doc is at bi-encoder rank 6–10). The k=10 → k=20 step gives **zero** improvement on this test set but adds ~12% latency. This tracks with re-ranker headroom expectations. **Recommended setting**: k=10.

### 3.3 Ablation C — Prompt engineering

| Metric | Permissive | Strict + missing-title guard |
|--------|----------:|------------------------------:|
| Refusals on 4 unanswerable queries | 0 / 4 | 3 / 4 |
| Hallucinated answers on unanswerable | 4 / 4 | 1 / 4 |
| Mean faithfulness, all queries | 0.900 | 0.936 |
| Mean faithfulness, excluding refusals | 0.900 | 0.924 |

**Note on faithfulness metric.** Reloading the Qwen2.5-7B RAGAS judge for two
separate prompt evaluations would require an additional runtime restart on T4.
As a proxy, I use the keyword-based custom faithfulness defined in CELL RAGAS-6,
which follows the same logical structure as RAGAS faithfulness (informative
tokens in the answer must appear in the retrieved context). The two summary
columns above are reported because the "all queries" version counts refusals as
1.0 (no claims to verify), which mechanically inflates the strict prompt's
average; the "excluding refusals" version measures faithfulness only on
substantive answers and is the fairer head-to-head comparison.

**Interpretation.** Even on the fair (refusal-excluded) comparison, the strict
prompt is more faithful (0.924 vs 0.900). The permissive prompt's 0.90 average
is pulled down by three specific queries: "What is The Completely Fictional
Banana Spaceship Movie about?" gets 0.0 faithfulness (Qwen-1.5B invents tokens
completely outside the retrieved context), "What is Dune 2021 about?" gets 0.67,
and the descriptive adversarial "A movie about a ship disaster and a romance
across social classes" gets 0.50. The strict prompt either refuses these or
sticks closer to the retrieved text.

**The critical point that faithfulness alone cannot catch.** For "Who directed
Oppenheimer?" (unanswerable), the permissive prompt scores **faithfulness =
1.0** — yet the answer "Jon Else" is factually wrong. Jon Else directed
*The Day After Trinity* (1981, the documentary that was retrieved), not the
2023 Oppenheimer film the user asked about. The metric is satisfied because
every informative token in the answer appears somewhere in the retrieved
context; the metric has no way to know the retrieved context is the wrong film.
The same silent-failure pattern shows up across the sequel-confusion category:
all four sequel queries score 1.0 faithfulness for both prompts, even though
"What is the plot of Terminator?" returns the plot of *Terminator Genisys*
(2015) rather than *The Terminator* (1984). This is exactly
a "confidently wrong, well-cited" failure mode, and it
empirically confirms that **inspecting `retrieved_titles` is mandatory**;
faithfulness on its own would have scored these failures perfectly.

**Note on Everything Everywhere All at Once.** The strict prompt unexpectedly
scores 0.0 on this query and does not register as a refusal. This is because
Ablation C ran *before* the improved `is_explicit_title_missing` guard in
CELL 30, which uses an extracted-then-validated title check rather than the
substring match of the original guard. The original guard fired on "Everything"
(an existing movie in the corpus), so the strict prompt was forced to answer
and produced tokens (drawn from parametric memory about the actual 2022 film)
that did not match the retrieved context. After CELL 30's fix is applied
(active in the final `run_query`), this query is correctly refused, bringing
strict hallucinations on unanswerable queries to 0/4 in the live system.

---

## 4. Retrieval Metrics on My 20-Query Test Set

| Mode | R@5 | R@10 | MRR | Answerable queries |
|------|----:|----:|----:|-------------------:|
| Bi-encoder only | 0.6875 | 0.7500 | 0.5911 | 16 |
| Bi-encoder + cross-encoder rerank | 0.7500 | 0.7500 | 0.5281 | 16 |
| Bi-encoder + title-aware rerank | **0.7500** | 0.7500 | **0.7188** | 16 |

**Discussion.** The vanilla cross-encoder helps R@5 (0.6875 → 0.7500) but actually **hurts MRR** (0.5911 → 0.5281). This is the sequel-confusion effect: the cross-encoder is trained on MS MARCO passages where lexical overlap is a strong signal, so it boosts longer/richer sequel descriptions to rank 1 ahead of the original film. The title-aware re-ranker — which adds a small bonus when a candidate's `title` is an exact substring match of the query — recovers the MRR (0.5281 → 0.7188) without losing R@5. This is a domain-specific trick that would not transfer to e.g. arxiv search, but for a movie corpus where users almost always type the title, it is a clear win.

---

## 5. Failure Diary Summary

Four cases are documented in CELL 33 with the required columns. Three include implemented before/after fixes:

- **"What is Avatar about?"** — title collision with *My Avatar and Me*. **Fix**: title-aware re-ranking. Before: top-1 was *My Avatar and Me*. After: top-1 is *Avatar*.
- **"Who directed Alien?"** — sequel confusion with *Alien 2: On Earth*. **Fix**: safer canonical-title matching that excludes *Alien³* etc. as substring matches. Before: Alien 2: On Earth at rank 1. After: Alien at rank 1, answer correctly attributes Ridley Scott.
- **"Who directed Oppenheimer?"** — corpus-absence hallucination (the LLM answered "Jon Else" from *The Day After Trinity*). **Fix**: explicit-title-missing guard before generation. After: clean refusal.
- **"A movie about dreams inside dreams..."** — semantic-drift retrieval miss (Dreamscape retrieved instead of Inception). **No fix implemented**, documented as a limitation of pure-bi-encoder retrieval on adversarial descriptive queries.

---

## 6. RAGAS Evaluation

**Metric mapping.** I evaluate with three RAGAS metrics: Faithfulness,
Context Relevance, and Answer Relevance. RAGAS 0.2.x deprecates the legacy
`context_relevancy` metric and recommends `context_precision` (LLM-judged
relevance of each retrieved chunk to the question, conditioned on the
ground-truth answer) as its successor. I use `context_precision` as the
context-relevance metric throughout this section. Faithfulness and Answer
Relevancy are taken from RAGAS as named.


The first attempt (CELL 38) failed because RAGAS defaults to OpenAI (`OPENAIError`). The second attempt (CELL 40) wrapped the 1.5B Qwen generator as a local judge — every job hit `TimeoutError` and all metrics returned NaN. Root cause: a 1.5B model cannot reliably emit the structured JSON RAGAS expects, and the `HuggingFacePipeline` defaulted to `max_length=20` which truncated every prompt before it could finish.

**Fix** (CELLs RAGAS-1 through RAGAS-7): save the full evaluation dataframe, **restart the runtime to free the generator + reranker + embedding model from VRAM**, then load Qwen2.5-7B-Instruct in 4-bit NF4 as the judge. Wrap the pipeline with `max_new_tokens=512`, `do_sample=False`, and pass `RunConfig(timeout=300, max_workers=1)` to RAGAS to serialise calls.

### Results (8-query subset, balanced across categories)

| Metric | Mean |
|--------|-----:|
| Faithfulness | 0.7143 |
| Answer Relevancy | 0.7128 |
| Context Precision | 0.1875 |
| Custom keyword faithfulness (full 20 queries) | 0.9900 |

### Discussion

- **Faithfulness 0.71**: 5 of 7 scored answers are perfectly faithful (1.0). The two zeros are diagnostic, not noise:
  - *"What is the plot of Terminator?"* (sequel_confusion): the system retrieved *Terminator Genisys* (2015) instead of *The Terminator* (1984) and described the wrong film's plot. The answer is technically grounded in the retrieved context, but not in the *correct* film — exactly the silent-failure mode the sequel-confusion category was designed to expose.
  - *"A movie about dreams inside dreams..."* (adversarial): the system refused because *Dreamscape* was retrieved instead of *Inception*, and the refusal does not match the gold-grounded reference.
- **Answer Relevancy 0.71**: high on substantive answers (0.94–1.00 across the board), pulled down by two refusals scoring 0.0 (refusals are technically not "relevant" answers to the question even when they are the correct behaviour).
- **Context Precision 0.19 (proxy for Context Relevance)**: this number looks
  alarming but is **a metric-setup artifact**, not a system failure. RAGAS
  `context_precision` is computed by asking the judge: "given this
  ground-truth answer, was each retrieved chunk useful for producing it?" My
  ground-truth field is a short template string (e.g. *"The answer should be
  grounded in: Inception."*) rather than the full expected answer text. The
  judge correctly observes that a 100-token plot block is not directly useful
  for producing the literal string *"...grounded in: Inception."*, so it
  scores most chunks as not-precise. The retrieval itself is fine — we
  separately measured R@5 = 0.75 with custom hit-based metrics on the same
  test set (Section 4) — so the low number reflects a labelling shortcut, not
  a retrieval failure. A proper fix would be writing full reference answers
  for all 20 queries, which is a labelling effort outside the current scope.
  I report the number honestly with this caveat rather than hide it or pick
  a different metric to make the table look better.
- **Custom keyword-based faithfulness 0.99**: every claim in every answer (across the full 20 queries, not just the RAGAS subset) consists of tokens that appear in the retrieved context. This confirms the generator is not hallucinating *new* tokens — when it gets the answer wrong, it gets it wrong by quoting the wrong retrieved film, not by inventing facts. This is consistent with the RAGAS faithfulness pattern and isolates the failure mode to **retrieval**, not generation.

**Weakest dimension**: faithfulness as a *metric* is misleading on this corpus.
Ablation C's faithfulness-per-prompt analysis (Section 3.3) shows that the
permissive prompt scores faithfulness = 1.0 on "Who directed Oppenheimer?"
while giving the factually wrong answer "Jon Else" (the director of the
retrieved 1981 documentary *The Day After Trinity*, not the 2023 film the user
asked about). All four sequel-confusion queries also score 1.0 even when the
wrong sequel is described. Faithfulness measures answer-context consistency,
not answer-correctness, and when the retrieved context is the wrong film, the
metric cannot tell. The system's weakest *behaviour* is therefore retrieval
on sequel-confusion and unanswerable-without-explicit-title queries, where the
retriever leaks the wrong document into a generator that then produces a
locally-faithful but globally-wrong answer. The title-aware rerank and the
explicit-title-missing guard (CELL 30) mitigate this for queries that name
the target movie directly, but the descriptive adversarial case remains
unsolved.

---

## 7. Run Evidence (Reproducibility)

- **Seed**: `SEED = 42`
- **Bi-encoder**: `BAAI/bge-small-en-v1.5`, batch_size 128, `normalize_embeddings=True`, fp32 vectors
- **Cross-encoder**: `cross-encoder/ms-marco-MiniLM-L-6-v2`
- **Generator**: `Qwen/Qwen2.5-1.5B-Instruct`, fp16, `do_sample=False`, `max_new_tokens=180`
- **RAGAS judge**: `Qwen/Qwen2.5-7B-Instruct`, 4-bit NF4 (bnb double-quant), `max_new_tokens=512`, `do_sample=False`
- **VectorDB**: FAISS `IndexFlatIP` over L2-normalized vectors
- **Retrieval defaults**: `k=20` (bi-encoder) → `n=5` (re-ranker)
**Persisted artifacts** (in `/content/drive/MyDrive/movie_rag_results/`): `end_to_end_outputs.csv`, `retrieval_metrics_summary.csv`, `title_aware_retrieval_details.csv`, `ablation_a_biencoder_size.csv`, `ablation_a_details_BAAI_bge-small-en-v1.5.csv`, `ablation_a_details_BAAI_bge-base-en-v1.5.csv`, `ablation_b_topk_sweep.csv`, `ablation_c_prompt_outputs.csv`, `ablation_c_summary.csv`, `ablation_c_faithfulness_per_prompt.csv`, `ablation_c_faithfulness_summary.csv`, `ragas_eval_df.pkl`, `ragas_eval_df_preview.csv`, `ragas_eval_df_with_custom_faithfulness.csv`, `ragas_results_qwen7b_judge.csv`, `ragas_summary.json`

**Runtime structure.** The notebook is organised into two sequential runtime
sessions on a single Colab T4, with one runtime restart between them. This is
not an oversight — it is a standard pattern for handling the GPU-memory
pressure that arises when the generator, re-ranker, embedding model, AND a
7B judge would otherwise need to be loaded simultaneously on a 16 GB T4:
save answers before evaluating, restart the runtime, then load only the
judge LLM without the generator LLM. Concretely:

- **Session 1**: CELLs 1 through 45 (data pipeline, embedding, indexing,
  retrieval, generation, all three ablations, retrieval metrics on the test
  set, the Failure Diary, and the run_query entrypoint). All
  evaluation data needed for RAGAS is persisted to Drive in CELL 34
  (`ragas_eval_df.pkl`).
- **Restart runtime** (Runtime → Restart session). This frees the bge-small
  bi-encoder, the MS-MARCO re-ranker, and the Qwen2.5-1.5B generator from
  VRAM.
- **Session 2**: CELLs RAGAS-1 through RAGAS-7. These re-mount Drive, load
  only the Qwen2.5-7B judge in 4-bit NF4 (~5.5 GB on T4), and run RAGAS on
  the persisted evaluation data. The custom keyword-based faithfulness in
  CELL RAGAS-6 covers the full 20-query test set and does not require the
  judge.

Each session runs top-to-bottom without intervention; the restart between them
is a one-click action and is the documented way to handle the T4 VRAM
constraint. All persisted artifacts (above) are in Drive so the second session
does not need to recompute anything from the first.

The `run_query` function is fully operational at the end of Session 1
(CELL 20 / 45). Loading the RAGAS judge in Session 2 is purely for
evaluation and is not part of the core pipeline.

---

## 8. Lessons Learned

1. **Title-as-text matters more than embedding size.** Putting the title and year directly inside `document_text` (rather than relying on metadata) had a bigger effect on retrieval quality than upgrading from `bge-small` to `bge-base`. Indexing what the user actually types into the query is the single highest-leverage design choice.
2. **Cross-encoders are not free.** The vanilla MS MARCO re-ranker actively hurt MRR on this corpus by promoting longer sequel descriptions over title-matched originals. A domain-aware tweak (title-aware boost) was needed to recover. Generic re-rankers should be benchmarked, not assumed to help.
3. **Prompt engineering > model size at the small-LLM scale.** Switching the Qwen2.5-1.5B prompt from permissive to strict cut hallucinations on unanswerable queries from 4/4 to 1/4 — far larger than any gain a 3B-vs-1.5B swap would have produced.
4. **RAGAS with a small judge silently fails.** A 1.5B judge produced all-NaN results in a way that looked like a library bug (TimeoutError) but was actually a structured-output capability gap. Always check that the judge can actually emit the expected JSON before trusting the metrics.
5. **Sequel confusion (and corpus-absence) is the worst failure mode in movie RAG,
   and faithfulness metrics cannot catch it.** The system is *confidently wrong,
   well-cited, and undetectable by metrics that only look at the answer field*.
   Ablation C's per-prompt faithfulness numbers show this empirically: all four
   sequel-confusion queries score 1.0 for both prompts, and the unanswerable
   "Who directed Oppenheimer?" scores 1.0 under the permissive prompt while
   returning "Jon Else" (the director of a retrieved 1981 Oppenheimer
   documentary, not the 2023 Christopher Nolan film). The only way to catch
   these failures is to inspect `retrieved_titles` directly — which is why the
   `run_query` interface returns it. Production RAG systems need this kind of
   provenance check baked into the contract, not as an optional debug output.